# Digital Twin of a Heated Water Tank: temperature + water level
**VTR UGE 21 · Digital Twin with Python · Final project**  (extends *A Python-Based Digital Twin for Temperature Monitoring and On–Off Control of a Heated Water Tank*)

**Honest scope:** this is a **software simulation**. The "asset" is a *simulated plant emulator*; there is no real tank, no real sensor, and no real measurement data. Calibration and validation use a clearly labelled **synthetic** benchmark dataset.

## How to run (Google Colab)
1. Upload this `.ipynb` to https://colab.research.google.com (File → Upload notebook), or open it from Google Drive.
2. Click **Runtime → Run all** (or press Ctrl+F9). Wait 1–2 minutes. No installation or account keys are needed (NumPy, SciPy, pandas, Matplotlib and ipywidgets are already in Colab).
3. Scroll to the bottom to the **Dashboard**. Change the sliders and press **Run / Simulate**. Use the tabs for results, intelligence-layer alerts, calibration/credibility and the data-flow log.
4. The last cell zips the `results/` and `data/` folders for download.

Outline: model → original baseline → tests → synthetic data + calibration → uncertainty → experiments → anomaly tests → scorecard → dashboard.

# Digital twin of a heated water tank (temperature + water level)
Software simulation only: the "asset" is a simulated plant emulator, NOT a real tank.
Each `# %%` marker below is one notebook cell (the .ipynb is generated from this file).

In [ ]:
import json
import math
from collections import deque
from dataclasses import dataclass, replace, asdict

import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
matplotlib.rcParams["figure.max_open_warning"] = 0

# ---------------------------------------------------------------- constants
RHO = 1000.0   # water density, kg/m^3 (constant: temperature effect on density ignored)
CP = 4200.0    # specific heat of water, J/(kg K). Chosen so C(h=0.5 m) = 420,000 J/K exactly
LV = 2.4e6     # latent heat of vaporisation, J/kg (about 50-60 degC)
T_MIN, T_MAX = 0.0, 100.0   # physical guard rails for liquid water at 1 atm, degC

## 1. Physics of the tank (two coupled states: T and h)

Mass balance:   dm/dt = m_in - m_out - m_evap,  with m = rho*A*h
Energy balance: d[(m*c + C_wall)*T]/dt = P*eta*u + m_in*c*T_in - m_out*c*T - m_evap*(c*T + Lv) - k(h)*(T - T_amb)

Coupling: (1) level h sets the thermal capacity C = rho*A*h*c + C_wall and the heat-loss
coefficient k(h); (2) temperature T sets the evaporation rate, which changes h.
With no flow, no evaporation, constant h and C_wall = 0, the update reduces to the original
report equation  T+ = T + dt/C * (P*u - k*(T - T_amb)).

In [ ]:
@dataclass(frozen=True)
class TankParams:
    """All physical parameters, with units. Frozen so a run cannot change them by accident."""
    area: float = 0.20            # m^2   tank cross-section (diameter about 0.50 m)
    h_min: float = 0.10           # m     lowest allowed water level (outlet pump cut-off)
    h_max: float = 0.80           # m     highest level; excess water overflows
    h_heater_min: float = 0.15    # m     heater dry-fire interlock level
    t_max_safe: float = 90.0      # degC  over-temperature heater cut-out
    c_wall: float = 0.0           # J/K   thermal mass of tank wall (0 in the twin, as in the report)
    k_fixed: float = 10.0         # W/K   heat loss through lid and base
    k_wall: float = 30.0          # W/K per m of water level (side-wall loss grows with wetted height)
    heater_power: float = 3000.0  # W     rated heater power P
    heater_eff: float = 1.0       # -     fraction of rated power reaching the water
    evap_coeff: float = 2.0e-8    # kg/(m^2 s Pa) mass-transfer coefficient of the free surface
    rel_humidity: float = 0.5     # -     room air relative humidity
    k_scale: float = 1.0          # -     multiplier on heat-loss coefficient (calibrated)
    evap_scale: float = 1.0       # -     multiplier on evaporation (calibrated)


def sat_pressure(T):
    """Saturation vapour pressure of water in Pa (Magnus formula, valid about 0-100 degC)."""
    return 610.94 * math.exp(17.625 * T / (T + 243.04))


def thermal_capacity(h, p):
    """Effective thermal capacity C(h) in J/K: water mass times c, plus wall mass."""
    return RHO * p.area * h * CP + p.c_wall


def heat_loss_coeff(h, p):
    """Heat-loss coefficient k(h) in W/K. At h = 0.5 m and k_scale = 1 it equals 25 W/K (original value)."""
    return p.k_scale * (p.k_fixed + p.k_wall * h)


def evap_rate(T, T_amb, p):
    """Evaporation mass flow in kg/s. Zero when surface vapour pressure is below that of room air."""
    dp = sat_pressure(T) - p.rel_humidity * sat_pressure(T_amb)
    return p.area * p.evap_coeff * p.evap_scale * max(0.0, dp)


def step_tank(T, h, u, q_in, q_out, T_in, T_amb, p, dt):
    """Advance the tank one time step (conservative forward Euler on mass m and energy U).

    T [degC], h [m], u heater command (0/1), q_in/q_out [L/min], T_in/T_amb [degC], dt [s].
    Returns a dict with the new state and every mass/energy term (used for conservation checks).
    """
    m = RHO * p.area * h
    m_min, m_max = RHO * p.area * p.h_min, RHO * p.area * p.h_max
    # Safety interlocks (dry-fire and over-temperature) can only switch the heater OFF.
    u_eff = 1.0 if (u >= 0.5 and h >= p.h_heater_min and T < p.t_max_safe) else 0.0

    m_in = q_in / 60.0 * RHO / 1000.0 * dt              # kg entering during the step
    m1 = m + m_in
    m_ev = min(evap_rate(T, T_amb, p) * dt, max(0.0, m1 - m_min))   # cannot evaporate below h_min
    m1 -= m_ev
    m_out = min(q_out / 60.0 * RHO / 1000.0 * dt, max(0.0, m1 - m_min))  # outlet limited at h_min
    m1 -= m_out
    m_ov = max(0.0, m1 - m_max)                          # overflow spills at tank temperature
    m1 -= m_ov

    e_heat = p.heater_power * p.heater_eff * u_eff * dt
    e_in = m_in * CP * T_in
    e_out = m_out * CP * T
    e_ev = m_ev * (CP * T + LV)
    e_ov = m_ov * CP * T
    e_loss = heat_loss_coeff(h, p) * (T - T_amb) * dt
    U = (m * CP + p.c_wall) * T
    U_new = U + e_heat + e_in - e_out - e_ev - e_ov - e_loss
    T_new = U_new / (m1 * CP + p.c_wall)
    clipped = False
    if not (T_MIN <= T_new <= T_MAX):                   # guard rail; should never trigger
        T_new, clipped = min(max(T_new, T_MIN), T_MAX), True
    return dict(T=T_new, h=m1 / (RHO * p.area), u_eff=u_eff, m_in=m_in, m_out=m_out, m_ev=m_ev,
                m_ov=m_ov, e_heat=e_heat, e_in=e_in, e_out=e_out, e_ev=e_ev, e_ov=e_ov,
                e_loss=e_loss, clipped=clipped)


def simulate_open_loop(p, u, q_in, q_out, T_in, T_amb, T0, h0, dt):
    """Run the tank model over given input arrays (length n). Returns T, h arrays of length n+1."""
    n = len(u)
    T = np.empty(n + 1)
    h = np.empty(n + 1)
    T[0], h[0] = T0, h0
    for k in range(n):
        s = step_tank(T[k], h[k], u[k], q_in[k], q_out[k], T_in[k], T_amb[k], p, dt)
        T[k + 1], h[k + 1] = s["T"], s["h"]
    return T, h


def stability_ok(p, h_lo, q_in_max, dt):
    """Check explicit-Euler stability: dt*(m_in*c + k)/C must stay well below 1."""
    C = thermal_capacity(h_lo, p)
    return dt * (q_in_max / 60.0 * CP + heat_loss_coeff(h_lo, p)) / C < 0.5


def original_one_state_baseline(seed=0, T0=20.0, target=55.0, T_amb=20.0, P=3000.0, C=420000.0,
                                k=25.0, dt=10.0, minutes=180, noise_sd=0.3):
    """Exact re-implementation of the ORIGINAL report simulation (one state, on/off control)."""
    rng = np.random.default_rng(seed)
    n = int(minutes * 60 / dt)
    T = T0
    Ts, us, ms = [], [], []
    for _ in range(n):
        meas = T + rng.normal(0, noise_sd)
        u = 1 if meas < target else 0
        T = T + dt / C * (P * u - k * (T - T_amb))
        Ts.append(T), us.append(u), ms.append(meas)
    return pd.DataFrame(dict(time_min=np.arange(1, n + 1) * dt / 60, T=Ts, T_meas=ms, u=us))

## 2. Scenario settings, plant emulator, message bus, digital twin
The plant emulator is a SIMULATED asset. Its parameters are hidden from the twin
(the twin only sees noisy telemetry). Messages travel as JSON text through a message bus.

In [ ]:
NOMINAL_TWIN = TankParams()   # parameter values from the original report (eta = 1, no wall mass)
# "True" simulated asset: differs from the nominal values so the twin has something to learn.
PLANT_PARAMS = TankParams(c_wall=15000.0, k_fixed=12.0, k_wall=28.0, heater_eff=0.93, evap_coeff=2.6e-8)

SPIKE_DURATION_S = 30.0
DEFAULT_ANOMALY_SIZE = {"spike": 5.0, "bias": 1.5, "heater_degradation": 0.4}


@dataclass
class Scenario:
    T0: float = 20.0            # degC   initial water temperature
    h0: float = 0.50            # m      initial water level
    target: float = 55.0        # degC   target temperature
    T_amb: float = 20.0         # degC   nominal ambient temperature
    heater_power: float = 3000.0  # W
    T_in: float = 15.0          # degC   inlet water temperature
    q_in: float = 0.1           # L/min  inlet flow
    q_out: float = 0.1          # L/min  outlet flow
    noise_sd: float = 0.3       # degC   temperature sensor noise (std dev)
    level_noise_sd: float = 0.002  # m   level sensor noise (std dev)
    duration_min: float = 180.0
    dt: float = 10.0            # s
    seed: int = 0
    controller: str = "twin_hysteresis"   # or "onoff_raw" (original rule on the raw reading)
    hysteresis: float = 1.0     # degC total band for twin controller
    ambient_drift_amp: float = 1.0  # degC unmodelled slow room-temperature drift (period 6 h)
    anomaly_type: str = "none"  # none | spike | bias | heater_degradation  (TEST scenarios)
    anomaly_start_min: float = 90.0
    anomaly_size: float = None  # None -> DEFAULT_ANOMALY_SIZE[type]


def validate_scenario(s):
    """Reject physically impossible inputs with a clear message (used by tests and dashboard)."""
    p = NOMINAL_TWIN
    checks = [
        (1.0 <= s.T0 <= 95.0, "initial temperature must be 1-95 degC"),
        (p.h_min <= s.h0 <= p.h_max, f"initial level must be {p.h_min}-{p.h_max} m"),
        (20.0 <= s.target <= 85.0, "target must be 20-85 degC"),
        (-10.0 <= s.T_amb <= 45.0, "ambient must be -10 to 45 degC"),
        (0.0 <= s.heater_power <= 10000.0, "heater power must be 0-10000 W"),
        (1.0 <= s.T_in <= 95.0, "inlet temperature must be 1-95 degC"),
        (0.0 <= s.q_in <= 50.0 and 0.0 <= s.q_out <= 50.0, "flows must be 0-50 L/min"),
        (0.0 <= s.noise_sd <= 5.0, "noise must be 0-5 degC"),
        (1.0 <= s.duration_min <= 2000.0, "duration must be 1-2000 min"),
        (s.dt > 0 and s.dt <= 30.0, "time step must be in (0, 30] s"),
        (s.controller in ("twin_hysteresis", "onoff_raw"), "unknown controller"),
        (s.anomaly_type in ("none", "spike", "bias", "heater_degradation"), "unknown anomaly type"),
    ]
    for ok, msg in checks:
        if not ok:
            raise ValueError("Invalid input: " + msg)
    if not stability_ok(p, p.h_min, s.q_in, s.dt):
        raise ValueError("Invalid input: time step too large for this inlet flow (Euler stability)")


class MessageBus:
    """Tiny software 'network' between asset and twin. Messages are JSON strings, so a real
    sensor/actuator link (MQTT, serial, HTTP) could replace this class without changing the twin."""

    def __init__(self):
        self.to_twin, self.to_asset = deque(), deque()
        self.telemetry_log, self.command_log = [], []

    def publish_telemetry(self, msg):          # Direction A: asset -> twin
        self.to_twin.append(json.dumps(msg))
        self.telemetry_log.append(dict(msg))

    def receive_telemetry(self):
        return json.loads(self.to_twin.popleft())

    def publish_command(self, msg):            # Direction B: twin -> asset
        self.to_asset.append(json.dumps(msg))
        self.command_log.append(dict(msg))

    def receive_command(self):
        return json.loads(self.to_asset.popleft())


class PlantEmulator:
    """Simulated physical asset (NOT a real tank). Holds the true state, produces noisy sensor
    telemetry, accepts actuator commands, and can inject test faults."""

    def __init__(self, scn, params, rng, bus):
        self.s, self.p, self.rng, self.bus = scn, params, rng, bus
        self.T, self.h, self.t = scn.T0, scn.h0, 0.0
        self.u_cmd, self.u_applied = 0, 0.0
        size = scn.anomaly_size if scn.anomaly_size is not None else DEFAULT_ANOMALY_SIZE.get(scn.anomaly_type)
        self.fault, self.fault_size = scn.anomaly_type, size

    def ambient_actual(self):
        return self.s.T_amb + self.s.ambient_drift_amp * math.sin(2 * math.pi * self.t / (6 * 3600.0))

    def _fault_active(self):
        return self.fault != "none" and self.t >= self.s.anomaly_start_min * 60.0

    def publish_telemetry(self):
        T_meas = self.T + self.rng.normal(0, self.s.noise_sd)
        h_meas = self.h + self.rng.normal(0, self.s.level_noise_sd)
        if self._fault_active():
            if self.fault == "spike" and self.t < self.s.anomaly_start_min * 60.0 + SPIKE_DURATION_S:
                T_meas += self.fault_size
            elif self.fault == "bias":
                T_meas += self.fault_size
        self.bus.publish_telemetry(dict(
            t=self.t, T_meas=T_meas, h_meas=h_meas, heater_state=self.u_applied,
            q_in=self.s.q_in, q_out=self.s.q_out, T_in=self.s.T_in))

    def receive_command(self):
        self.u_cmd = self.bus.receive_command()["heater_cmd"]

    def advance(self):
        p = self.p
        if self._fault_active() and self.fault == "heater_degradation":
            p = replace(p, heater_eff=p.heater_eff * (1.0 - self.fault_size))
        out = step_tank(self.T, self.h, self.u_cmd, self.s.q_in, self.s.q_out, self.s.T_in,
                        self.ambient_actual(), p, self.s.dt)
        self.T, self.h, self.u_applied = out["T"], out["h"], out["u_eff"]
        self.t += self.s.dt
        self.last = out

## 3. Intelligence layer
**A. Predictive monitoring** - `predict_time_to_target` runs the twin's physics forward.
**B. Anomaly detection** - three complementary statistical detectors on the twin's residuals.

In [ ]:
def predict_time_to_target(T0, h0, target, q_in, q_out, T_in, T_amb, p, dt=10.0, horizon_s=8 * 3600):
    """Predict seconds until the water reaches `target` with the heater held ON, using the twin's model
    from the current state and current operating conditions. Returns dict(status, seconds, reason)."""
    if T0 >= target:
        return dict(status="at_target", seconds=0.0, reason="already at or above target")
    T, h = T0, h0
    n = int(horizon_s / dt)
    for k in range(n):
        s = step_tank(T, h, 1, q_in, q_out, T_in, T_amb, p, dt)
        Tn = s["T"]
        if Tn >= target:
            frac = (target - T) / max(Tn - T, 1e-12)
            return dict(status="ok", seconds=(k + frac) * dt, reason="")
        if abs(Tn - T) < 1e-6 and k > 20:
            break
        T, h = Tn, s["h"]
    if h < p.h_heater_min:
        why = f"heater inhibited: water level {h:.2f} m is below the dry-fire limit {p.h_heater_min} m"
    else:
        why = f"target not reachable: temperature levels off near {T:.1f} degC under current flow/power/loss"
    return dict(status="unavailable", seconds=float("nan"), reason=why)


class AnomalyDetector:
    """Three detectors on the twin's prediction residuals (innovations):
    1. spike    - |z| of the one-step innovation exceeds z_spike (point outliers, sensor glitches)
    2. drift    - two-sided CUSUM on z (small persistent bias that starts suddenly)
    3. rate     - measured temperature change over a 10-min window (mean of last 10 readings minus mean of
                  first 10) vs the change the model expects (unexpected heating rate, e.g. degraded heater)
    sigma_innov = sigma*sqrt(2/(2-K)) is the exact steady-state innovation std of a fixed-gain observer
    (gain K) when the model is perfect, so z is standardised and thresholds have a stated false-alarm rate
    (z=4 -> about 6e-5 per sample for Gaussian noise)."""

    def __init__(self, sigma, gain, dt, z_spike=4.0, cusum_k=0.5, cusum_h=8.0, window=60,
                 z_rate=4.0, rate_margin=0.0005, block=10):
        self.sig_innov = max(sigma, 1e-6) * math.sqrt(2.0 / (2.0 - gain))
        self.sigma, self.dt = max(sigma, 1e-6), dt
        self.z_spike, self.ck, self.ch, self.z_rate = z_spike, cusum_k, cusum_h, z_rate
        self.window, self.rate_margin, self.block = window, rate_margin, block
        self.reject_run = 0
        idx = np.arange(window) - (window - 1) / 2.0
        self.idx, self.sxx = idx, float(np.sum(idx ** 2))
        self.meas, self.exp_rates = deque(maxlen=window), deque(maxlen=window)
        self.sp, self.sm = 0.0, 0.0
        self.active, self.alerts = {}, []

    def _raise(self, kind, t, value, thr, msg, flagged):
        if flagged and kind not in self.active:
            a = dict(type=kind, t_start=t, t_end=None, value=value, threshold=thr, message=msg)
            self.alerts.append(a)
            self.active[kind] = a
        elif not flagged and kind in self.active:
            self.active.pop(kind)["t_end"] = t

    def update(self, t, T_meas, innov, exp_rate_per_s):
        """Feed one sample. Returns True if the sample is flagged as a spike (so the twin can reject it)."""
        z = innov / self.sig_innov
        spike = abs(z) > self.z_spike
        self._raise("spike", t, z, self.z_spike,
                    f"Sensor reading deviates {z:+.1f} sigma from the twin's prediction", spike)
        zc = max(-self.z_spike, min(self.z_spike, z))
        self.sp, self.sm = max(0.0, self.sp + zc - self.ck), max(0.0, self.sm - zc - self.ck)
        drift = max(self.sp, self.sm) > self.ch
        self._raise("drift", t, max(self.sp, self.sm), self.ch,
                    "Persistent offset between sensor and twin prediction (CUSUM alarm)", drift)
        if drift:
            self.sp = self.sm = 0.0
        self.meas.append(T_meas)
        self.exp_rates.append(exp_rate_per_s * self.dt)
        zr = 0.0
        if len(self.meas) == self.window:
            m = self.block
            meas = np.asarray(self.meas)
            d_meas = float(meas[-m:].mean() - meas[:m].mean())          # measured net change (degC)
            lo = int(round((m + 1) / 2.0))                              # intervals between the two block centres
            d_exp = float(np.sum(list(self.exp_rates)[lo:lo + self.window - m]))   # change the model expects
            sig = math.sqrt(2.0 * self.sigma ** 2 / m + (self.rate_margin * (self.window - m)) ** 2)
            zr = (d_meas - d_exp) / sig
        self._raise("rate", t, zr, self.z_rate,
                    f"Heating rate over the last {self.window * self.dt / 60:.0f} min differs from the model ({zr:+.1f} sigma)",
                    abs(zr) > self.z_rate)
        return spike, z, zr


def make_recommendations(metrics, alerts, prediction):
    """Rule-based advice generated from the twin's results (supporting feature)."""
    r = []
    if prediction and prediction.get("status") == "unavailable":
        r.append("Target not reachable as set: " + prediction["reason"] + ". Raise heater power, reduce flows or lower the target.")
    if metrics.get("n_switches", 0) > 60:
        r.append("Heater switches often; increase the hysteresis band to reduce wear.")
    if metrics.get("min_h", 1) <= NOMINAL_TWIN.h_min + 0.01:
        r.append("Water level reached the minimum limit; reduce outlet flow or increase inlet flow.")
    if metrics.get("overflow_kg", 0) > 0:
        r.append("Tank overflowed; reduce inlet flow or increase outlet flow.")
    kinds = {a["type"] for a in alerts}
    if "spike" in kinds:
        r.append("Spike alert: a reading was rejected by the twin; inspect the temperature sensor and wiring.")
    if "drift" in kinds:
        r.append("Drift alert: sensor may be biased or the model no longer matches; recalibrate or check the sensor.")
    if "rate" in kinds:
        r.append("Heating-rate alert: heater output differs from the model; inspect the heater/element.")
    if not r:
        r.append("No action needed: the asset behaves as the twin predicts.")
    return r


class DigitalTwin:
    """Digital twin: receives telemetry (Direction A), keeps a model-based state estimate,
    runs the intelligence layer, and sends heater commands + setpoint (Direction B)."""

    GAIN_T, GAIN_H = 0.2, 0.2   # fixed observer gains (0 = trust model only, 1 = trust sensor only)

    def __init__(self, scn, params, bus, predict_every_s=300.0):
        self.s, self.p, self.bus = scn, params, bus
        self.T_hat = self.h_hat = self.T_prior = self.h_prior = None
        self.u_prev, self.exp_rate = 0, 0.0
        self.detector = AnomalyDetector(scn.noise_sd, self.GAIN_T, scn.dt)
        self.predict_every, self.predictions = predict_every_s, []
        self.z_spike_now = self.z_now = self.zr_now = 0.0
        self.rejected = False
        self.msg = None

    def process_telemetry(self):
        m = self.msg = self.bus.receive_telemetry()
        if self.T_prior is None:                      # first sample: initialise from sensors
            self.T_hat, self.h_hat = m["T_meas"], m["h_meas"]
            self.innov_T = self.innov_h = 0.0
            self.rejected = False
        else:
            self.innov_T, self.innov_h = m["T_meas"] - self.T_prior, m["h_meas"] - self.h_prior
            flagged, self.z_now, self.zr_now = self.detector.update(m["t"], m["T_meas"], self.innov_T, self.exp_rate)
            self.reject_run = self.reject_run + 1 if flagged else 0
            # reject an outlier, but accept readings again if >5 in a row are 'outliers' (the model may be wrong)
            self.rejected = flagged and self.reject_run <= 5
            self.T_hat = self.T_prior if self.rejected else self.T_prior + self.GAIN_T * self.innov_T
            self.h_hat = self.h_prior + self.GAIN_H * self.innov_h
        if m["t"] % self.predict_every < 1e-6:
            pr = predict_time_to_target(self.T_hat, self.h_hat, self.s.target, m["q_in"], m["q_out"],
                                        m["T_in"], self.s.T_amb, self.p, self.s.dt)
            pr["t"] = m["t"]
            self.predictions.append(pr)

    def send_command(self):
        m, s = self.msg, self.s
        if s.controller == "onoff_raw":
            u, ref, reason = (1 if m["T_meas"] < s.target else 0), m["T_meas"], "ON below target (raw reading)"
        else:
            lo, hi = s.target - s.hysteresis / 2.0, s.target + s.hysteresis / 2.0
            u = 1 if self.T_hat <= lo else 0 if self.T_hat >= hi else self.u_prev
            if s.hysteresis == 0:
                u = 1 if self.T_hat < s.target else 0
            ref, reason = self.T_hat, "twin estimate with hysteresis"
        if self.h_hat < self.p.h_heater_min:
            u, reason = 0, "heater inhibited: low water level"
        self.u_prev = u
        self.bus.publish_command(dict(t=m["t"], heater_cmd=u, target=s.target, mode=s.controller,
                                      control_variable=ref, reason=reason))
        self.u_sent = u

    def predict_next(self):
        m = self.msg
        out = step_tank(self.T_hat, self.h_hat, self.u_sent, m["q_in"], m["q_out"], m["T_in"],
                        self.s.T_amb, self.p, self.s.dt)
        self.exp_rate = (out["T"] - self.T_hat) / self.s.dt
        self.T_prior, self.h_prior = out["T"], out["h"]

In [ ]:
def get_twin_params(heater_power=3000.0, calibrated=True):
    """Twin parameters: calibrated (if calibration has been run) or nominal report values."""
    if calibrated:
        ensure_calibrated()
    base = replace(NOMINAL_TWIN, heater_power=heater_power)
    if calibrated and _CALIBRATED:
        base = replace(base, **_CALIBRATED)
    return base


_CALIBRATED = {}


def set_calibrated_params(d):
    _CALIBRATED.clear()
    _CALIBRATED.update({k: float(v) for k, v in d.items()})


def run_twin_simulation(scn, twin_params=None, plant_params=None):
    """Closed-loop simulation: plant emulator <-> message bus <-> digital twin <-> controller."""
    validate_scenario(scn)
    twin_params = twin_params or get_twin_params(scn.heater_power)
    plant_params = plant_params or replace(PLANT_PARAMS, heater_power=scn.heater_power)
    n = int(round(scn.duration_min * 60 / scn.dt))
    rng = np.random.default_rng(scn.seed)
    bus = MessageBus()
    plant, twin = PlantEmulator(scn, plant_params, rng, bus), DigitalTwin(scn, twin_params, bus)
    rows = []
    for k in range(n + 1):
        row = dict(time_s=plant.t, T_true=plant.T, h_true=plant.h, T_amb_actual=plant.ambient_actual())
        plant.publish_telemetry()          # Direction A  (asset -> twin)
        twin.process_telemetry()
        row.update(T_meas=twin.msg["T_meas"], h_meas=twin.msg["h_meas"], T_hat=twin.T_hat, h_hat=twin.h_hat,
                   T_pred=twin.T_prior if twin.T_prior is not None else twin.T_hat,
                   h_pred=twin.h_prior if twin.h_prior is not None else twin.h_hat,
                   innov_T=twin.innov_T, innov_h=twin.innov_h, z_T=twin.z_now, z_rate=twin.zr_now,
                   rejected=twin.rejected, heater_state=twin.msg["heater_state"])
        if k < n:
            twin.send_command()            # Direction B  (twin -> asset)
            plant.receive_command()
            row["heater_cmd"] = twin.u_sent
            twin.predict_next()
            plant.advance()
        else:
            row["heater_cmd"] = np.nan
        rows.append(row)
    df = pd.DataFrame(rows)
    df["time_min"] = df.time_s / 60.0
    res = SimResult(scn, twin_params, plant_params, df, bus, twin)
    return res


class SimResult:
    """Holds everything produced by one run, plus summary metrics."""

    def __init__(self, scn, twin_params, plant_params, df, bus, twin):
        self.scn, self.twin_params, self.plant_params, self.df, self.bus = scn, twin_params, plant_params, df, bus
        self.alerts = [dict(a, t_start_min=a["t_start"] / 60, t_end_min=None if a["t_end"] is None else a["t_end"] / 60)
                       for a in twin.detector.alerts]
        self.predictions = pd.DataFrame(twin.predictions)
        self.telemetry_log = pd.DataFrame(bus.telemetry_log)
        self.command_log = pd.DataFrame(bus.command_log)
        self.metrics = self._metrics()
        self.recommendations = make_recommendations(self.metrics, self.alerts,
                                                    self.predictions.iloc[0].to_dict() if len(self.predictions) else None)

    def _metrics(self):
        df, s = self.df, self.scn
        body = df.iloc[:-1]
        hit = df.index[df.T_true >= s.target]
        t_hit = float(df.time_min[hit[0]]) if len(hit) else None
        pred0 = self.predictions.iloc[0].to_dict() if len(self.predictions) else {}
        pred_min = pred0.get("seconds", float("nan")) / 60 if pred0 else float("nan")
        cmd = body.heater_cmd.to_numpy()
        return dict(
            final_T=float(df.T_true.iloc[-1]), final_h=float(df.h_true.iloc[-1]),
            min_T=float(df.T_true.min()), max_T=float(df.T_true.max()), min_h=float(df.h_true.min()),
            max_h=float(df.h_true.max()),
            heater_on_pct=float(100 * np.mean(df.heater_state.iloc[1:])),   # state applied during each of the n steps
            heater_on_min=float(np.sum(df.heater_state.iloc[1:]) * s.dt / 60),
            n_switches=int(np.sum(np.abs(np.diff(cmd)) > 0)),
            time_to_target_actual_min=t_hit, predicted_ttt_initial_min=pred_min,
            ttt_error_min=(pred_min - t_hit) if (t_hit is not None and pred_min == pred_min) else None,
            mae_T_pred=mae(df.T_meas.iloc[1:], df.T_pred.iloc[1:]), rmse_T_pred=rmse(df.T_meas.iloc[1:], df.T_pred.iloc[1:]),
            mae_h_pred=mae(df.h_meas.iloc[1:], df.h_pred.iloc[1:]), rmse_h_pred=rmse(df.h_meas.iloc[1:], df.h_pred.iloc[1:]),
            mae_T_est_true=mae(df.T_hat, df.T_true), rmse_T_est_true=rmse(df.T_hat, df.T_true),
            rmse_T_meas_true=rmse(df.T_meas, df.T_true),
            n_alerts=len(self.alerts), n_rejected=int(df.rejected.sum()),
            evap_kg=float(self._sum_diag("m_ev")), overflow_kg=float(self._sum_diag("m_ov")),
            all_finite=bool(np.isfinite(df.select_dtypes(include=[float]).drop(columns=["heater_cmd"]).to_numpy()).all()))

    def _sum_diag(self, key):
        # Re-run the plant deterministically from the log to total a mass term (cheap, noise-free).
        p, s = self.plant_params, self.scn
        T, h, tot = s.T0, s.h0, 0.0
        for k, u in enumerate(self.df.heater_cmd.iloc[:-1]):
            amb = s.T_amb + s.ambient_drift_amp * math.sin(2 * math.pi * k * s.dt / (6 * 3600.0))
            pp = p
            if s.anomaly_type == "heater_degradation" and k * s.dt >= s.anomaly_start_min * 60:
                size = s.anomaly_size if s.anomaly_size is not None else DEFAULT_ANOMALY_SIZE["heater_degradation"]
                pp = replace(p, heater_eff=p.heater_eff * (1 - size))
            o = step_tank(T, h, u, s.q_in, s.q_out, s.T_in, amb, pp, s.dt)
            T, h, tot = o["T"], o["h"], tot + o[key]
        return tot


def mae(a, b):
    """Mean absolute error."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.mean(np.abs(a - b)))


def rmse(a, b):
    """Root mean squared error."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(np.sqrt(np.mean((a - b) ** 2)))


def heater_on_fraction(u):
    """Fraction of steps with heater ON."""
    return float(np.mean(np.asarray(u, float)))

## 4. Synthetic benchmark dataset and parameter calibration
No real measurements exist, so a **synthetic** reference dataset is generated from an independently
coded benchmark (continuous ODE solved with SciPy, different vapour-pressure formula, extra wall mass,
sensor lag, drifting ambient). The twin is then fitted to the CALIBRATION part and tested on the
VALIDATION part. This demonstrates the procedure; it is not calibration against a real tank.

In [ ]:
BENCHMARK_TRUE = dict(heater_eff=0.93, k_fixed=12.0, k_wall=28.0, c_wall=15000.0, evap_coeff=2.6e-8,
                      rel_humidity=0.5, sensor_tau=15.0, heater_power=3000.0, area=0.20)
# schedule rows: (start_min, end_min, heater u, q_in L/min, q_out L/min, T_in degC)
CAL_SCHEDULE = [(0, 50, 1, 0.0, 0.0, 15.0), (50, 110, 0, 0.0, 0.0, 15.0), (110, 145, 1, 4.5, 3.0, 15.0),
                (145, 180, 0, 0.0, 2.0, 15.0), (180, 215, 1, 3.0, 2.0, 25.0), (215, 240, 1, 2.0, 2.0, 25.0)]
CAL_FRACTION = 0.6
BOUNDS = dict(heater_eff=(0.5, 1.0), k_scale=(0.5, 2.0), evap_scale=(0.1, 5.0))
SIGMA_T, SIGMA_H = 0.3, 0.002
DATA_COLUMNS = ["time_s", "heater_cmd", "q_in_lpm", "q_out_lpm", "T_in_C", "T_amb_C", "T_meas_C", "h_meas_m"]


def _buck(T):
    """Buck (1981) saturation vapour pressure, Pa - deliberately different from the twin's Magnus formula."""
    return 611.21 * math.exp((18.678 - T / 234.5) * (T / (257.14 + T)))


def generate_benchmark_dataset(seed=42, schedule=CAL_SCHEDULE, T0=20.0, h0=0.5, dt=10.0, true=BENCHMARK_TRUE):
    """Independent benchmark (SciPy ODE solver) + realistic sensor noise. SYNTHETIC data."""
    from scipy.integrate import solve_ivp
    rng = np.random.default_rng(seed)
    A = true["area"]
    amb = lambda t: 20.0 + 1.0 * math.sin(2 * math.pi * t / (6 * 3600.0))
    y = [RHO * A * h0, T0, T0]
    ts, Ts, Tsens, hs, rows_in = [], [], [], [], []
    for (a, b, u, qi, qo, Tin) in schedule:
        def rhs(t, y):
            m, T, Tl = y
            h = m / (RHO * A)
            kh = true["k_fixed"] + true["k_wall"] * h
            ev = A * true["evap_coeff"] * max(0.0, _buck(T) - true["rel_humidity"] * _buck(amb(t)))
            din, dout = qi / 60.0, qo / 60.0
            dm = din - dout - ev
            dT = (true["heater_power"] * true["heater_eff"] * u + din * CP * (Tin - T) - ev * LV
                  - kh * (T - amb(t))) / (m * CP + true["c_wall"])
            return [dm, dT, (T - Tl) / true["sensor_tau"]]
        grid = np.arange(a * 60.0, b * 60.0 + 1e-9, dt)
        sol = solve_ivp(rhs, (grid[0], grid[-1]), y, t_eval=grid, rtol=1e-9, atol=1e-9, max_step=5.0)
        y = sol.y[:, -1]
        sl = slice(0, None) if not ts else slice(1, None)
        ts += list(sol.t[sl]); Ts += list(sol.y[1][sl]); Tsens += list(sol.y[2][sl])
        hs += list(sol.y[0][sl] / (RHO * A))
        rows_in += [(u, qi, qo, Tin)] * len(sol.t[sl])
    # inputs at row k apply on [t_k, t_k+dt): shift so the last row copies the final segment
    n = len(ts)
    ts = np.array(ts)
    inp = np.array(rows_in, float)
    # (rows_in[k] is the segment that ENDS at t_k; recompute properly from the schedule instead)
    for k, t in enumerate(ts):
        for (a, b, u, qi, qo, Tin) in schedule:
            if a * 60.0 <= t < b * 60.0 or (t >= schedule[-1][1] * 60.0 and b == schedule[-1][1]):
                inp[k] = (u, qi, qo, Tin)
                break
    df = pd.DataFrame(dict(time_s=ts, heater_cmd=inp[:, 0].astype(int), q_in_lpm=inp[:, 1], q_out_lpm=inp[:, 2],
                           T_in_C=inp[:, 3],
                           T_amb_C=[amb(t) + rng.normal(0, 0.1) for t in ts],
                           T_meas_C=np.array(Tsens) + rng.normal(0, SIGMA_T, n),
                           h_meas_m=np.array(hs) + rng.normal(0, SIGMA_H, n)))
    df["T_benchmark_truth_C"] = Ts           # truth columns exist only because the data are synthetic
    df["h_benchmark_truth_m"] = hs
    cut = CAL_FRACTION * ts[-1]
    df["split"] = np.where(ts <= cut, "calibration", "validation")
    df["data_origin"] = "SYNTHETIC_BENCHMARK_NOT_REAL_MEASUREMENTS"
    return df


def load_reference_csv(path):
    """Load a REAL dataset with columns time_s, heater_cmd, q_in_lpm, q_out_lpm, T_in_C, T_amb_C,
    T_meas_C, h_meas_m on a uniform 10 s grid. Returns (df, report). Short gaps (<=3 samples) are linearly
    interpolated; the report lists everything that was changed so it can be documented."""
    df = pd.read_csv(path)
    missing_cols = [c for c in DATA_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"CSV is missing columns: {missing_cols}")
    df = df.sort_values("time_s").reset_index(drop=True)
    steps = np.diff(df.time_s.to_numpy())
    if not np.allclose(steps, steps[0]) or abs(steps[0] - 10.0) > 1e-6:
        raise ValueError("time_s must be uniform with 10 s spacing (resample your data first)")
    n_nan = int(df[DATA_COLUMNS].isna().sum().sum())
    df[DATA_COLUMNS] = df[DATA_COLUMNS].interpolate(limit=3, limit_direction="both")
    if df[DATA_COLUMNS].isna().any().any():
        raise ValueError("gaps longer than 3 samples remain; fix or remove them")
    if not (df.T_meas_C.between(0, 100).all() and df.h_meas_m.between(0, 1.5).all()):
        raise ValueError("temperature must be 0-100 degC and level 0-1.5 m (check units)")
    df["split"] = np.where(df.time_s <= CAL_FRACTION * df.time_s.iloc[-1], "calibration", "validation")
    df["data_origin"] = "REAL_USER_DATA"
    return df, dict(rows=len(df), missing_values_filled=n_nan)


def _twin_from_theta(theta, base):
    return replace(base, heater_eff=theta[0], k_scale=theta[1], evap_scale=theta[2])


def simulate_twin_on_dataset(df, p, T0=20.0, h0=0.5, dt=10.0):
    """Free-running twin prediction over the whole record, using only the recorded INPUTS and initial state."""
    n = len(df) - 1
    return simulate_open_loop(p, df.heater_cmd.to_numpy()[:n], df.q_in_lpm.to_numpy()[:n], df.q_out_lpm.to_numpy()[:n],
                              df.T_in_C.to_numpy()[:n], df.T_amb_C.to_numpy()[:n], T0, h0, dt)


def calibration_workflow(df=None, T0=20.0, h0=0.5, apply=True):
    """Fit heater efficiency, heat-loss scale and evaporation scale of the twin to the calibration split
    (weighted nonlinear least squares, bounded). Evaluate on the validation split. Returns a dict."""
    from scipy.optimize import least_squares
    if df is None:
        df = generate_benchmark_dataset()
    base = replace(NOMINAL_TWIN, heater_power=3000.0)
    cal = (df.split == "calibration").to_numpy()

    def residuals(theta):
        T, h = simulate_twin_on_dataset(df, _twin_from_theta(theta, base), T0, h0)
        rT = (df.T_meas_C.to_numpy() - T)[cal] / SIGMA_T
        rh = (df.h_meas_m.to_numpy() - h)[cal] / SIGMA_H
        return np.concatenate([rT, rh])

    names = ["heater_eff", "k_scale", "evap_scale"]
    lo, hi = zip(*[BOUNDS[n] for n in names])
    x0 = [1.0, 1.0, 1.0]
    sol = least_squares(residuals, x0, bounds=(lo, hi), x_scale=[0.1, 0.1, 0.5], diff_step=1e-3)
    th = sol.x
    dof = len(sol.fun) - len(th)
    s2 = float(np.sum(sol.fun ** 2) / dof)
    cov = s2 * np.linalg.inv(sol.jac.T @ sol.jac)
    se = np.sqrt(np.diag(cov))
    tw_cal = _twin_from_theta(th, base)
    tw_nom = base
    out = dict(theta=dict(zip(names, map(float, th))), se=dict(zip(names, map(float, se))), cov=cov, s2=s2,
               bounds=BOUNDS, df=df, base=base, params=tw_cal, cost=float(sol.cost), nfev=int(sol.nfev))
    for label, p in (("calibrated", tw_cal), ("nominal", tw_nom)):
        T, h = simulate_twin_on_dataset(df, p, T0, h0)
        out[label] = dict(T=T, h=h)
        for sp in ("calibration", "validation"):
            m = (df.split == sp).to_numpy()
            out[label][sp] = dict(
                mae_T=mae(df.T_meas_C[m], T[m]), rmse_T=rmse(df.T_meas_C[m], T[m]),
                mae_h=mae(df.h_meas_m[m], h[m]), rmse_h=rmse(df.h_meas_m[m], h[m]))
    if apply:
        set_calibrated_params(out["theta"])
    return out


_DATASET_CACHE = {}


def ensure_calibrated():
    """Run the synthetic-data calibration once if it has not been run (so the twin always has parameters)."""
    if not _CALIBRATED:
        calibration_workflow(apply=True)


def reference_values(true=BENCHMARK_TRUE):
    """Benchmark values in the twin's parameterisation. k_scale and evap_scale are 'equivalent' values at
    h = 0.5 m, because the benchmark and twin loss formulas differ slightly in their level dependence."""
    k_true = true["k_fixed"] + true["k_wall"] * 0.5
    k_twin = NOMINAL_TWIN.k_fixed + NOMINAL_TWIN.k_wall * 0.5
    return dict(heater_eff=true["heater_eff"], k_scale=k_true / k_twin,
                evap_scale=true["evap_coeff"] / NOMINAL_TWIN.evap_coeff)

## 5. Uncertainty and sensitivity

In [ ]:
def quick_closed_loop(scn, p, seed):
    """Fast plant-only loop with the ORIGINAL on/off rule on the noisy reading (used for Monte Carlo)."""
    rng = np.random.default_rng(seed)
    n = int(scn.duration_min * 60 / scn.dt)
    T, h, on, t_hit, T60 = scn.T0, scn.h0, 0, None, None
    for k in range(n):
        u = 1 if T + rng.normal(0, scn.noise_sd) < scn.target else 0
        o = step_tank(T, h, u, scn.q_in, scn.q_out, scn.T_in, scn.T_amb, p, scn.dt)
        T, h, on = o["T"], o["h"], on + o["u_eff"]
        if t_hit is None and T >= scn.target:
            t_hit = (k + 1) * scn.dt / 60
        if k + 1 == int(3600 / scn.dt):
            T60 = T
    return dict(t_target_min=t_hit if t_hit is not None else np.nan, T60=T60, final_T=T, final_h=h, on_pct=100 * on / n)


def uncertainty_analysis(cal, n=200, seed=1):
    """(i) parameter uncertainty band on the validation experiment; (ii) sensor-noise spread; (iii) both."""
    rng = np.random.default_rng(seed)
    df, base = cal["df"], cal["base"]
    th = np.array([cal["theta"][k] for k in ("heater_eff", "k_scale", "evap_scale")])
    lo, hi = zip(*[BOUNDS[k] for k in ("heater_eff", "k_scale", "evap_scale")])
    draws = np.clip(rng.multivariate_normal(th, cal["cov"], n), lo, hi)
    band_T = np.array([simulate_twin_on_dataset(df, _twin_from_theta(d, base))[0] for d in draws])
    scn = Scenario(ambient_drift_amp=0.0)
    p0 = cal["params"]
    noise_only = [quick_closed_loop(replace(scn), p0, 1000 + i) for i in range(n)]
    both = [quick_closed_loop(replace(scn), _twin_from_theta(draws[i], base), 2000 + i) for i in range(n)]
    summ = lambda L, key: (float(np.nanmean([r[key] for r in L])), float(np.nanstd([r[key] for r in L])),
                           float(np.nanpercentile([r[key] for r in L], 5)), float(np.nanpercentile([r[key] for r in L], 95)))
    keys = ["t_target_min", "T60", "final_T", "final_h", "on_pct"]
    return dict(band_T=band_T, draws=draws, time_s=df.time_s.to_numpy(),
                noise_only={k: summ(noise_only, k) for k in keys}, both={k: summ(both, k) for k in keys}, n=n)


def sensitivity_analysis(cal, rel=0.10):
    """One-at-a-time +/-10 % changes; noise-free closed loop on the default scenario."""
    scn = Scenario(noise_sd=0.0, ambient_drift_amp=0.0)
    p0 = cal["params"]
    base_out = quick_closed_loop(scn, p0, 0)
    rows = []
    for name in ["heater_eff", "k_scale", "evap_scale", "heater_power"]:
        for sign in (-1, +1):
            p = replace(p0, **{name: getattr(p0, name) * (1 + sign * rel)})
            o = quick_closed_loop(scn, p, 0)
            rows.append(dict(parameter=name, change=f"{sign * rel * 100:+.0f}%", t_target_min=o["t_target_min"],
                             final_h_m=o["final_h"], on_pct=o["on_pct"]))
    for name, attr in [("T_amb (+/-2 degC)", "T_amb"), ("T_in (+/-10%)", "T_in"), ("flows (+/-10%)", None)]:
        for sign in (-1, +1):
            if name.startswith("T_amb"):
                sc = replace(scn, T_amb=scn.T_amb + sign * 2.0)
            elif attr == "T_in":
                sc = replace(scn, T_in=scn.T_in * (1 + sign * rel))
            else:
                sc = replace(scn, q_in=scn.q_in * (1 + sign * rel), q_out=scn.q_out * (1 + sign * rel))
            o = quick_closed_loop(sc, p0, 0)
            rows.append(dict(parameter=name, change=f"{sign * rel * 100:+.0f}%" if attr != "T_amb" else f"{sign * 2:+.0f} degC",
                             t_target_min=o["t_target_min"], final_h_m=o["final_h"], on_pct=o["on_pct"]))
    return dict(base=base_out, table=pd.DataFrame(rows))

## 6. Plots

In [ ]:
def plot_run(res, title_suffix=""):
    """Six-panel dashboard figure for one simulation run."""
    df, s = res.df, res.scn
    fig, ax = plt.subplots(3, 2, figsize=(13, 10))
    a = ax[0, 0]
    a.plot(df.time_min, df.T_meas, color="orange", lw=0.6, alpha=0.7, label="Simulated sensor reading")
    a.plot(df.time_min, df.T_true, color="tab:blue", label="Asset true temperature (simulation)")
    a.plot(df.time_min, df.T_hat, color="tab:green", ls="--", lw=1.2, label="Twin estimate")
    a.axhline(s.target, color="red", ls=":", label=f"Target {s.target:g} °C")
    a.set(title="Water temperature", xlabel="Time (min)", ylabel="Temperature (°C)"); a.legend(fontsize=7)
    a = ax[0, 1]
    a.plot(df.time_min, df.h_meas, color="orange", lw=0.6, alpha=0.7, label="Level sensor")
    a.plot(df.time_min, df.h_true, color="tab:blue", label="True level")
    a.plot(df.time_min, df.h_hat, color="tab:green", ls="--", label="Twin estimate")
    a.axhline(NOMINAL_TWIN.h_max, color="k", ls=":", lw=0.8); a.axhline(NOMINAL_TWIN.h_min, color="k", ls=":", lw=0.8)
    a.set(title="Water level (dotted: min/max)", xlabel="Time (min)", ylabel="Level (m)"); a.legend(fontsize=7)
    a = ax[1, 0]
    a.step(df.time_min[:-1], df.heater_cmd[:-1], where="post", label="Command from twin")
    a.step(df.time_min, df.heater_state, where="post", ls="--", color="tab:red", lw=0.8, label="Heater state reported by asset")
    a.set(title="Heater ON/OFF", xlabel="Time (min)", ylabel="0 = OFF, 1 = ON", yticks=[0, 1]); a.legend(fontsize=7)
    a = ax[1, 1]
    a.plot(df.time_min, df.T_meas, color="orange", lw=0.6, label="Asset reading")
    a.plot(df.time_min, df.T_pred, color="tab:purple", lw=0.9, label="Twin prediction")
    a.set(title="Asset reading vs twin prediction", xlabel="Time (min)", ylabel="Temperature (°C)"); a.legend(fontsize=7)
    a = ax[2, 0]
    a.plot(df.time_min, df.innov_T, lw=0.6, label="Temperature residual (°C)")
    sig = res.twin_params and s.noise_sd * math.sqrt(2 / (2 - DigitalTwin.GAIN_T))
    a.axhline(4 * sig, color="r", ls=":", lw=0.8); a.axhline(-4 * sig, color="r", ls=":", lw=0.8, label="±4σ spike limit")
    for al in res.alerts:
        a.axvline(al["t_start_min"], color={"spike": "red", "drift": "purple", "rate": "brown"}[al["type"]], alpha=0.7,
                  label=f"alert: {al['type']}")
    h_, l_ = a.get_legend_handles_labels(); u_ = dict(zip(l_, h_)); a.legend(u_.values(), u_.keys(), fontsize=7)
    a.set(title="Prediction residual (measurement - prediction)", xlabel="Time (min)", ylabel="°C")
    a = ax[2, 1]
    pr = res.predictions
    if len(pr):
        ok = pr[pr.status == "ok"]
        a.plot(ok.t / 60, ok.seconds / 60, "o-", ms=3, label="Predicted time still needed (min)")
        if res.metrics["time_to_target_actual_min"] is not None:
            tt = res.metrics["time_to_target_actual_min"]
            xs = np.array([0, tt]); a.plot(xs, tt - xs, "k--", lw=0.8, label="Actual time remaining")
    a.set(title="Predicted time to reach target", xlabel="Time (min)", ylabel="Minutes"); a.legend(fontsize=7)
    fig.suptitle("Simulated asset + digital twin" + title_suffix, fontsize=12)
    fig.tight_layout()
    return fig


def plot_calibration(cal):
    df = cal["df"]
    t = df.time_s / 60
    cut = df.time_s[df.split == "calibration"].max() / 60
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    for j, (col, key, lab) in enumerate([("T_meas_C", "T", "Temperature (°C)"), ("h_meas_m", "h", "Level (m)")]):
        a = ax[0, j]
        a.plot(t, df[col], color="orange", lw=0.5, label="Synthetic measurement")
        a.plot(t, cal["nominal"][key], color="gray", ls=":", label="Twin, nominal parameters")
        a.plot(t, cal["calibrated"][key], color="tab:blue", label="Twin, calibrated")
        a.axvline(cut, color="k", ls="--"); a.text(cut + 1, a.get_ylim()[0], " validation →", fontsize=8)
        a.set(title=f"{lab.split(' ')[0]}: fit (left of dashed line) and validation", xlabel="Time (min)", ylabel=lab); a.legend(fontsize=7)
        r = df[col].to_numpy() - cal["calibrated"][key]
        a = ax[1, j]
        a.plot(t, r, lw=0.6); a.axhline(0, color="k", lw=0.5); a.axvline(cut, color="k", ls="--")
        a.set(title=f"Residuals (measurement - calibrated twin), {lab.split(' ')[0].lower()}", xlabel="Time (min)", ylabel=lab)
    fig.suptitle("Calibration on SYNTHETIC reference data", fontsize=12)
    fig.tight_layout()
    return fig


def plot_uncertainty(unc, cal):
    df = cal["df"]
    t = unc["time_s"] / 60
    q5, q50, q95 = np.percentile(unc["band_T"], [5, 50, 95], axis=0)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].fill_between(t, q5, q95, alpha=0.4, label="5-95 % band, parameter uncertainty")
    ax[0].plot(t, q50, label="Median prediction")
    ax[0].plot(t, df.T_meas_C, color="orange", lw=0.4, label="Synthetic measurement")
    ax[0].set(title="Prediction band from calibration covariance", xlabel="Time (min)", ylabel="Temperature (°C)"); ax[0].legend(fontsize=7)
    m = unc["noise_only"]["t_target_min"], unc["both"]["t_target_min"]
    ax[1].bar(["Sensor noise only", "Noise + parameters"], [m[0][0], m[1][0]], yerr=[m[0][1], m[1][1]], capsize=6)
    ax[1].set(title="Time to reach 55 °C (mean ± 1 std, 200 runs)", ylabel="Minutes")
    fig.tight_layout()
    return fig


def plot_architecture(path=None):
    fig, ax = plt.subplots(figsize=(13, 6.5)); ax.axis("off"); ax.set_xlim(0, 13); ax.set_ylim(0, 7)
    def box(x, y, w, h, text, fc):
        ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec="k", lw=1.2)); ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=9)
    def arrow(x1, y1, x2, y2, text, c, dy=0.18):
        ax.annotate("", (x2, y2), (x1, y1), arrowprops=dict(arrowstyle="->", lw=2, color=c))
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + dy, text, ha="center", fontsize=7.5, color=c)
    box(0.2, 2.2, 3.4, 3.0, "SIMULATED ASSET\n(plant emulator, not a real tank)\n\nStates: T(t), h(t)\nSensors: temperature, level\nActuator: heater (+ interlocks)\nInputs: flows, inlet T, ambient", "#d9ead3")
    box(5.2, 2.2, 3.2, 3.0, "DIGITAL TWIN\n\nTwin model (calibrated)\nState estimator\nResiduals", "#cfe2f3")
    box(9.4, 3.9, 3.4, 2.2, "INTELLIGENCE LAYER\nA: time-to-target prediction\nB: anomaly detection\n(spike, CUSUM, heating rate)\n+ recommendations", "#fff2cc")
    box(9.4, 0.8, 3.4, 2.2, "CONTROLLER\nON/OFF with hysteresis on twin\nestimate (or original raw rule)\nSafety: low-level inhibit", "#f4cccc")
    arrow(3.6, 4.6, 5.2, 4.6, "A: telemetry (JSON)\nT, h, heater,\nflows", "tab:green", 0.2)
    arrow(5.2, 2.8, 3.6, 2.8, "B: commands (JSON)\nheater ON/OFF,\ntarget", "tab:red", -0.75)
    arrow(8.4, 4.7, 9.4, 4.7, "state, residuals", "k", 0.15)
    arrow(9.4, 3.2, 8.4, 3.0, "decision", "k", -0.4)
    arrow(11.1, 3.9, 11.1, 3.0, "alerts, setpoint", "k", 0.0)
    ax.text(6.5, 6.6, "Bidirectional data flow (software message bus)", ha="center", fontsize=13, weight="bold")
    ax.text(6.5, 0.2, "Real sensors/actuators would replace the plant emulator; the JSON message format stays the same.", ha="center", fontsize=9, style="italic")
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    return fig

## 7. Dashboard backend and Colab widgets

In [ ]:
CONTROL_SPECS = [  # (key, label with unit, min, max, step, default)
    ("T0", "Initial water temperature (°C)", 1, 90, 1, 20),
    ("h0", "Initial water level (m)", 0.10, 0.80, 0.01, 0.50),
    ("target", "Target temperature (°C)", 25, 85, 1, 55),
    ("T_amb", "Ambient temperature (°C)", 0, 40, 1, 20),
    ("heater_power", "Heater power (W)", 500, 5000, 100, 3000),
    ("T_in", "Inlet water temperature (°C)", 5, 60, 1, 15),
    ("q_in", "Inlet flow (L/min)", 0, 10, 0.1, 0.1),
    ("q_out", "Outlet flow (L/min)", 0, 10, 0.1, 0.1),
    ("noise_sd", "Sensor noise std dev (°C)", 0, 1.5, 0.05, 0.3),
    ("duration_min", "Simulation duration (min)", 30, 480, 10, 180),
]


def scenario_from_values(v):
    """Turn a dict of dashboard values into a validated Scenario."""
    s = Scenario(**{k: v[k] for k in v if k in Scenario.__dataclass_fields__})
    validate_scenario(s)
    return s


def status_text(res):
    m, s = res.metrics, res.scn
    if m["time_to_target_actual_min"] is None:
        phase = f"The water has NOT reached the {s.target:g} °C target (final {m['final_T']:.1f} °C)."
    else:
        phase = (f"The asset reached {s.target:g} °C after {m['time_to_target_actual_min']:.0f} min and ended at "
                 f"{m['final_T']:.1f} °C.")
    pr = res.predictions.iloc[0].to_dict() if len(res.predictions) else {}
    ptxt = (f"At t = 0 the twin predicted {pr['seconds'] / 60:.0f} min." if pr.get("status") == "ok"
            else f"Time-to-target estimate unavailable: {pr.get('reason', '')}")
    alert = f"{m['n_alerts']} alert(s) raised." if m["n_alerts"] else "No anomaly alerts."
    return (f"{phase} {ptxt} Level ended at {m['final_h']:.3f} m (range {m['min_h']:.3f}-{m['max_h']:.3f} m). "
            f"Heater ON {m['heater_on_pct']:.1f}% of the time with {m['n_switches']} switch events. {alert}")


def metrics_table(res):
    m = res.metrics
    rows = [("Final water temperature (°C)", f"{m['final_T']:.2f}"), ("Final water level (m)", f"{m['final_h']:.3f}"),
            ("Heater ON time (%)", f"{m['heater_on_pct']:.1f}"), ("Heater switch events", m["n_switches"]),
            ("Actual time to target (min)", "not reached" if m["time_to_target_actual_min"] is None else f"{m['time_to_target_actual_min']:.1f}"),
            ("Twin initial prediction of time to target (min)", "unavailable" if m["predicted_ttt_initial_min"] != m["predicted_ttt_initial_min"] else f"{m['predicted_ttt_initial_min']:.1f}"),
            ("Prediction vs reading, temperature MAE / RMSE (°C)", f"{m['mae_T_pred']:.3f} / {m['rmse_T_pred']:.3f}"),
            ("Prediction vs reading, level MAE / RMSE (m)", f"{m['mae_h_pred']:.4f} / {m['rmse_h_pred']:.4f}"),
            ("Twin estimate vs true temperature, RMSE (°C) [simulation only]", f"{m['rmse_T_est_true']:.3f}"),
            ("Alerts / readings rejected by twin", f"{m['n_alerts']} / {m['n_rejected']}"),
            ("Evaporated water (kg)", f"{m['evap_kg']:.3f}"), ("Overflow (kg)", f"{m['overflow_kg']:.2f}")]
    return pd.DataFrame(rows, columns=["Metric", "Value"])


def dashboard_update(values):
    """Backend of the dashboard: dict of control values -> results. Used by the widget callback and by tests."""
    try:
        scn = scenario_from_values(values)
    except ValueError as e:
        return dict(ok=False, status=str(e))
    res = run_twin_simulation(scn)
    alerts = [f"[{a['type'].upper()}] t = {a['t_start_min']:.1f} min: {a['message']}" for a in res.alerts] or ["No alerts."]
    return dict(ok=True, scenario=scn, result=res, status=status_text(res), metrics=metrics_table(res), alerts=alerts,
                recommendations=res.recommendations, figure=plot_run(res))


def calibration_summary_table(cal):
    ref = reference_values()
    rows = []
    for k in ("heater_eff", "k_scale", "evap_scale"):
        est, se = cal["theta"][k], cal["se"][k]
        rows.append(dict(parameter=k, benchmark=round(ref[k], 3), estimate=round(est, 3), std_err=round(se, 3),
                         error_pct=round(100 * (est - ref[k]) / ref[k], 1)))
    return pd.DataFrame(rows)


def calibration_metrics_table(cal):
    rows = []
    for lab in ("nominal", "calibrated"):
        for sp in ("calibration", "validation"):
            r = cal[lab][sp]
            rows.append(dict(twin=lab, split=sp, MAE_T_C=round(r["mae_T"], 3), RMSE_T_C=round(r["rmse_T"], 3),
                             MAE_h_mm=round(1000 * r["mae_h"], 2), RMSE_h_mm=round(1000 * r["rmse_h"], 2)))
    return pd.DataFrame(rows)


def build_dashboard(widgets=None, display_fn=None, clear_fn=None, credibility_df=None):
    """Create the interactive dashboard (ipywidgets). Call display(build_dashboard()) in Colab."""
    if widgets is None:
        import ipywidgets as widgets
    if display_fn is None or clear_fn is None:
        try:
            from IPython.display import display as _d, clear_output as _c
        except ImportError:
            _d, _c = (lambda *a, **k: None), (lambda *a, **k: None)
        display_fn, clear_fn = display_fn or _d, clear_fn or _c
    ensure_calibrated()
    ctl = {}
    for key, label, lo, hi, step, default in CONTROL_SPECS:
        ctl[key] = widgets.FloatSlider(value=default, min=lo, max=hi, step=step, description=label,
                                       style={"description_width": "230px"}, layout=widgets.Layout(width="560px"))
    ctl["controller"] = widgets.Dropdown(options=[("Twin estimate + hysteresis", "twin_hysteresis"), ("Original on/off on raw reading", "onoff_raw")],
                                         value="twin_hysteresis", description="Controller", style={"description_width": "230px"})
    ctl["anomaly_type"] = widgets.Dropdown(options=[("None", "none"), ("TEST: sensor spike", "spike"), ("TEST: sensor bias", "bias"),
                                                    ("TEST: heater degradation", "heater_degradation")], value="none",
                                           description="Inject synthetic anomaly (test)", style={"description_width": "230px"})
    ctl["anomaly_start_min"] = widgets.FloatSlider(value=90, min=10, max=400, step=5, description="Anomaly start (min)",
                                                   style={"description_width": "230px"}, layout=widgets.Layout(width="560px"))
    ctl["seed"] = widgets.IntText(value=0, description="Random seed", style={"description_width": "230px"})
    run_button = widgets.Button(description="Run / Simulate", button_style="success")
    status = widgets.HTML("<i>Press Run / Simulate.</i>")
    out_main, out_intel, out_cal, out_log = widgets.Output(), widgets.Output(), widgets.Output(), widgets.Output()
    tabs = widgets.Tab(children=[out_main, out_intel, out_cal, out_log])
    for i, t in enumerate(["Results", "Intelligence layer", "Calibration & credibility", "Data-flow log"]):
        tabs.set_title(i, t)

    def on_run(_=None):
        vals = {k: w.value for k, w in ctl.items()}
        r = dashboard_update(vals)
        for o in (out_main, out_intel, out_cal, out_log):
            with o:
                clear_fn(wait=True)
        if not r["ok"]:
            status.value = f"<b style='color:red'>{r['status']}</b>"
            return
        status.value = "<b>Status:</b> " + r["status"] + ("<br><b style='color:#b45f06'>TEST SCENARIO: a synthetic anomaly was injected.</b>" if vals["anomaly_type"] != "none" else "")
        res = r["result"]
        with out_main:
            display_fn(r["metrics"]); display_fn(r["figure"]); plt.close(r["figure"])
        with out_intel:
            print("PREDICTIVE MONITORING (time to reach target)")
            pr = res.predictions
            print(pr[["t", "status", "seconds", "reason"]].head(8).to_string(index=False) if len(pr) else "no prediction")
            print("\nANOMALY ALERTS");  [print(" ", a) for a in r["alerts"]]
            print("\nRECOMMENDATIONS");  [print(" -", x) for x in r["recommendations"]]
        with out_cal:
            cal = calibration_workflow(apply=False) if "cal" not in _DATASET_CACHE else _DATASET_CACHE["cal"]
            _DATASET_CACHE["cal"] = cal
            print("Calibration used SYNTHETIC reference data (not a real tank).")
            display_fn(calibration_summary_table(cal)); display_fn(calibration_metrics_table(cal))
            if credibility_df is not None:
                display_fn(credibility_df[["Area", "Status"]])
        with out_log:
            print("Direction A: asset -> twin (telemetry, first 5 and last 2 messages)")
            display_fn(pd.concat([res.telemetry_log.head(5), res.telemetry_log.tail(2)]))
            print("Direction B: twin -> asset (commands, first 5 and last 2 messages)")
            display_fn(pd.concat([res.command_log.head(5), res.command_log.tail(2)]))

    run_button.on_click(on_run)
    box = widgets.VBox([widgets.HTML("<h3>Heated water tank: simulated asset + digital twin (software simulation)</h3>")]
                       + list(ctl.values()) + [run_button, status, tabs])
    box.run_button, box.controls, box.status_widget, box.on_run = run_button, ctl, status, on_run
    return box

## 8. Evidence collection, credibility scorecard and requirement checklist
`collect_evidence` reruns every experiment and returns the numbers used in the report, so the
notebook, the report tables and the scorecard always come from the same code.

In [ ]:
REPORT_BASELINE = dict(final_T=55.3, heater_on_pct=67.6, heater_on_min=121.7, T30=32.2, T60=43.2, T90=53.0)

SCENARIOS = {
    "S1 Default (flows 0.1 L/min)": Scenario(),
    "S2 No flow (closest to original)": Scenario(q_in=0.0, q_out=0.0),
    "S3 Original raw on/off controller": Scenario(controller="onoff_raw"),
    "S4 Drain to minimum level": Scenario(q_in=0.5, q_out=2.0),
    "S5 Fill to overflow": Scenario(q_in=3.0, q_out=0.5, h0=0.6),
}
ANOMALY_TESTS = {"spike": Scenario(anomaly_type="spike", anomaly_start_min=90),
                 "bias": Scenario(anomaly_type="bias", anomaly_start_min=90),
                 "heater_degradation": Scenario(anomaly_type="heater_degradation", anomaly_start_min=30)}


def _save(fig, outdir, name):
    if outdir:
        import os
        os.makedirs(os.path.join(outdir, "figures"), exist_ok=True)
        fig.savefig(os.path.join(outdir, "figures", name), dpi=130, bbox_inches="tight")


def collect_evidence(outdir=None, tests_df=None, n_mc=200, n_false_alarm_seeds=40, show_progress=print):
    """Run all experiments. If outdir is given, figures/tables/CSV files are written there."""
    import os
    ev = {}
    say = show_progress or (lambda *a: None)
    say("1/7 original baseline"); base = original_one_state_baseline()
    ev["baseline"] = dict(final_T=float(base["T"].iloc[-1]), heater_on_pct=float(100 * base["u"].mean()),
                          heater_on_min=float(base["u"].sum() * 10 / 60), T30=float(base["T"].iloc[179]),
                          T60=float(base["T"].iloc[359]), T90=float(base["T"].iloc[539]),
                          T120=float(base["T"].iloc[719]), min_T=float(base["T"].min()), max_T=float(base["T"].max()),
                          mean_T=float(base["T"].mean()))
    say("2/7 calibration")
    ds = generate_benchmark_dataset()
    cal = calibration_workflow(ds, apply=True)
    ev["cal"] = cal
    ev["cal_params_table"], ev["cal_metrics_table"] = calibration_summary_table(cal), calibration_metrics_table(cal)
    say("3/7 uncertainty + sensitivity")
    ev["unc"] = uncertainty_analysis(cal, n=n_mc)
    ev["sens"] = sensitivity_analysis(cal)
    say("4/7 scenarios")
    ev["runs"] = {name: run_twin_simulation(s) for name, s in SCENARIOS.items()}
    ev["scenario_table"] = pd.DataFrame([dict(Scenario=n, **{
        "Final T (°C)": round(r.metrics["final_T"], 2), "Final level (m)": round(r.metrics["final_h"], 3),
        "Min level (m)": round(r.metrics["min_h"], 3), "Heater ON (%)": round(r.metrics["heater_on_pct"], 1),
        "Switch events": r.metrics["n_switches"],
        "Time to target (min)": None if r.metrics["time_to_target_actual_min"] is None else round(r.metrics["time_to_target_actual_min"], 1),
        "Predicted at t=0 (min)": None if r.metrics["predicted_ttt_initial_min"] != r.metrics["predicted_ttt_initial_min"] else round(r.metrics["predicted_ttt_initial_min"], 1),
        "Overflow (kg)": round(r.metrics["overflow_kg"], 1), "Evaporated (kg)": round(r.metrics["evap_kg"], 3)})
        for n, r in ev["runs"].items()])
    say("5/7 anomaly tests")
    ev["anom_runs"] = {k: run_twin_simulation(s) for k, s in ANOMALY_TESTS.items()}
    rows = []
    for k, r in ev["anom_runs"].items():
        onset = ANOMALY_TESTS[k].anomaly_start_min * 60
        kinds = {"spike": ("spike", "drift", "rate"), "bias": ("spike", "drift"), "heater_degradation": ("rate",)}[k]
        hit = [a for a in r.alerts if a["type"] in kinds and a["t_start"] >= onset]
        first = min(hit, key=lambda a: a["t_start"]) if hit else None
        rows.append(dict(test_anomaly=k, onset_min=ANOMALY_TESTS[k].anomaly_start_min,
                         detected=bool(first), first_alert=None if not first else first["type"],
                         delay_s=None if not first else first["t_start"] - onset,
                         alerts_total=len(r.alerts), readings_rejected=r.metrics["n_rejected"]))
    ev["anomaly_table"] = pd.DataFrame(rows)
    fa = [len(run_twin_simulation(Scenario(seed=s)).alerts) for s in range(n_false_alarm_seeds)]
    ev["false_alarms"] = dict(runs=n_false_alarm_seeds, total=int(sum(fa)), per_run=sum(fa) / n_false_alarm_seeds)
    ttt_err = [run_twin_simulation(Scenario(seed=s)).metrics["ttt_error_min"] for s in range(10)]
    ev["ttt_error"] = dict(mean=float(np.mean(ttt_err)), max_abs=float(np.max(np.abs(ttt_err))))
    say("6/7 figures")
    figs = {}
    figs["fig1_architecture.png"] = plot_architecture()
    d0 = base
    r1 = ev["runs"]["S1 Default (flows 0.1 L/min)"]
    r2 = ev["runs"]["S2 No flow (closest to original)"]
    f, a = plt.subplots(figsize=(9, 4.5))
    a.plot(d0.time_min, d0["T"], label="Original one-state model (report)")
    a.plot(r2.df.time_min, r2.df.T_true, label="Extended model S2 (no flow)")
    a.plot(r1.df.time_min, r1.df.T_true, label="Extended model S1 (0.1 L/min flows)")
    a.axhline(55, color="r", ls=":"); a.set(xlabel="Time (min)", ylabel="Water temperature (°C)", title="Original vs extended model"); a.legend()
    figs["fig2_baseline_vs_extended.png"] = f
    figs["fig3_default_run.png"] = plot_run(r1, " - scenario S1")
    figs["fig4_drain_run.png"] = plot_run(ev["runs"]["S4 Drain to minimum level"], " - scenario S4 (drain)")
    figs["fig5_overflow_run.png"] = plot_run(ev["runs"]["S5 Fill to overflow"], " - scenario S5 (overflow)")
    f, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=False)
    for a, (k, r) in zip(ax, ev["anom_runs"].items()):
        a.plot(r.df.time_min, r.df.T_meas, color="orange", lw=0.5, label="Sensor reading")
        a.plot(r.df.time_min, r.df.T_true, label="True temperature"); a.plot(r.df.time_min, r.df.T_hat, "g--", lw=1, label="Twin estimate")
        a.axvline(ANOMALY_TESTS[k].anomaly_start_min, color="k", ls="--", label="Injected TEST anomaly")
        for al in r.alerts:
            a.axvline(al["t_start_min"], color={"spike": "red", "drift": "purple", "rate": "brown"}[al["type"]], alpha=0.5, lw=1)
        a.set(title=f"TEST scenario: {k} (coloured lines = alerts: red spike, purple drift, brown rate)", ylabel="°C"); a.legend(fontsize=7, loc="lower right")
    ax[-1].set_xlabel("Time (min)"); f.tight_layout()
    figs["fig6_anomaly_tests.png"] = f
    figs["fig7_calibration.png"] = plot_calibration(cal)
    figs["fig8_uncertainty.png"] = plot_uncertainty(ev["unc"], cal)
    tab = ev["sens"]["table"]; base_t = ev["sens"]["base"]["t_target_min"]
    f, a = plt.subplots(figsize=(8, 4.5))
    labels = tab.parameter + " " + tab.change
    a.barh(labels, tab.t_target_min - base_t); a.axvline(0, color="k")
    a.set(xlabel=f"Change in time to reach target (min), baseline {base_t:.1f} min", title="One-at-a-time sensitivity (±10 %)")
    f.tight_layout(); figs["fig9_sensitivity.png"] = f
    rs, rr = ev["runs"]["S1 Default (flows 0.1 L/min)"], ev["runs"]["S3 Original raw on/off controller"]
    f, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    for a, r, t in zip(ax, (rr, rs), ("Original rule on raw reading", "Twin estimate + 1 °C hysteresis")):
        m = r.df.time_min > 100
        a.step(r.df.time_min[m], r.df.heater_cmd[m], where="post"); a.set(title=f"{t}: {r.metrics['n_switches']} switches", xlabel="Time (min)", yticks=[0, 1])
    f.tight_layout(); figs["fig10_controller_comparison.png"] = f
    ev["figs"] = figs
    ev["tests"] = tests_df
    ev["scorecard"] = build_scorecard(ev)
    if outdir:
        say("7/7 writing files")
        os.makedirs(os.path.join(outdir, "tables"), exist_ok=True)
        for n, fg in figs.items():
            _save(fg, outdir, n)
        ddir = os.path.join(os.path.dirname(outdir.rstrip("/")) or ".", "data")
        os.makedirs(ddir, exist_ok=True)
        ds.to_csv(os.path.join(ddir, "synthetic_reference_dataset.csv"), index=False)
        for nm in ("cal_params_table", "cal_metrics_table", "scenario_table", "anomaly_table", "scorecard"):
            ev[nm].to_csv(os.path.join(outdir, "tables", nm + ".csv"), index=False)
        tab.to_csv(os.path.join(outdir, "tables", "sensitivity_table.csv"), index=False)
        r1.telemetry_log.to_csv(os.path.join(outdir, "tables", "telemetry_log_S1.csv"), index=False)
        r1.command_log.to_csv(os.path.join(outdir, "tables", "command_log_S1.csv"), index=False)
        r1.df.to_csv(os.path.join(outdir, "tables", "run_S1_timeseries.csv"), index=False)
        if tests_df is not None:
            tests_df.to_csv(os.path.join(outdir, "tables", "test_results.csv"), index=False)
    for fg in figs.values():
        plt.close(fg)
    return ev


def build_scorecard(ev):
    """Completed credibility scorecard. Numbers are pulled from the evidence of this run."""
    cal, m = ev["cal"], ev["cal"]
    th, se, ref = cal["theta"], cal["se"], reference_values()
    cv, nv = cal["calibrated"]["validation"], cal["nominal"]["validation"]
    t = ev.get("tests")
    n_pass = int((t.result == "PASS").sum()) if t is not None else None
    n_tot = len(t) if t is not None else None
    tests_txt = f"{n_pass}/{n_tot} automated tests passed in this run" if t is not None else "tests not run in this session"
    unc, fa, tt = ev["unc"], ev["false_alarms"], ev["ttt_error"]
    an = ev["anomaly_table"]
    det = f"{int(an.detected.sum())}/{len(an)} injected test anomalies detected (delays: " + ", ".join(f"{r.test_anomaly} {r.delay_s:.0f} s" for r in an.itertuples() if r.detected) + ")"
    allpass = t is not None and n_pass == n_tot
    rows = [
        ("Conceptual consistency and assumptions",
         "Model states, equations and assumptions are physically interpretable and reduce to the original equation",
         "Section 3 of report; step_tank(); tests 04, 09, 10, 11",
         "Reduces to original equation to 1e-9 (test 04); mass and energy balances close; analytic mixing/cooling/heating limits matched",
         "Demonstrated" if allpass else "Partially demonstrated",
         "Lumped energy and mass balance with level-dependent capacity/loss and temperature-dependent evaporation; limiting cases have closed-form answers.",
         "Uniform temperature, constant density/specific heat, assumed evaporation coefficient and humidity; no stratification; not checked against a physical tank."),
        ("Numerical implementation and verification",
         "Correct steps, bounds, conservation, finite values, reproducibility",
         "tests/test_digital_twin_tank.py; results/tables/test_results.csv",
         tests_txt + "; forward Euler with dt = 10 s (stability checked by stability_ok)",
         "Demonstrated" if allpass else "Partially demonstrated",
         "Computational verification only: tests confirm the code solves the stated equations, including level limits and interlocks.",
         "Verification is not validation; no time-step convergence study beyond closed-form checks at dt = 10 s."),
        ("Parameter calibration",
         "Unknown parameters fitted with documented method, bounds and separate validation data",
         "calibration_workflow(); data/synthetic_reference_dataset.csv; Figure 7",
         f"Bounded least squares: heater_eff {th['heater_eff']:.3f} (benchmark {ref['heater_eff']:.2f}), k_scale {th['k_scale']:.3f} (benchmark-equivalent {ref['k_scale']:.2f}), evap_scale {th['evap_scale']:.2f} ± {se['evap_scale']:.2f} (benchmark-equivalent {ref['evap_scale']:.2f})",
         "Partially demonstrated",
         "Procedure is complete and leak-free (test 35) but uses SYNTHETIC data; estimates absorb unmodelled wall mass and sensor lag, so efficiency and loss are biased by several percent.",
         "No real measurements available, so the twin is NOT calibrated to a real tank. Reported standard errors are optimistic (assume independent errors)."),
        ("Validation against reference data",
         "Predictions compared with data not used for fitting",
         "Figure 7; results/tables/cal_metrics_table.csv",
         f"Validation split: temperature RMSE {cv['rmse_T']:.3f} °C (MAE {cv['mae_T']:.3f}) vs {nv['rmse_T']:.3f} °C for nominal twin; level RMSE {1000 * cv['rmse_h']:.2f} mm (sensor noise 2 mm)",
         "Partially demonstrated",
         "Synthetic-data validation on a held-out period with different flow regimes; error is close to the sensor-noise floor (0.3 °C).",
         "Real-world validation: NOT demonstrated. The synthetic benchmark shares the same physics family, so agreement is optimistic."),
        ("Uncertainty and sensitivity",
         "Effect of sensor noise and parameter uncertainty on predictions",
         "uncertainty_analysis(), sensitivity_analysis(); Figures 8-9",
         f"Time to 55 °C: {unc['noise_only']['t_target_min'][0]:.1f} ± {unc['noise_only']['t_target_min'][1]:.1f} min (noise only), {unc['both']['t_target_min'][0]:.1f} ± {unc['both']['t_target_min'][1]:.1f} min (noise + parameters), {unc['n']} runs",
         "Partially demonstrated",
         "Monte Carlo from the calibration covariance and ±10 % one-at-a-time sensitivity are implemented and plotted.",
         "Model-form (structural) uncertainty and input uncertainty (flows, ambient) are not propagated; covariance is optimistic."),
        ("Intelligence-layer behaviour",
         "At least two distinct intelligence elements execute and are evaluated",
         "predict_time_to_target(); AnomalyDetector; tests 21-28; Figure 6",
         f"Time-to-target error at t=0: mean {tt['mean']:+.1f} min, max |error| {tt['max_abs']:.1f} min (10 seeds); {det}; false alarms {fa['total']} in {fa['runs']} normal runs ({fa['per_run']:.2f} per run)",
         "Demonstrated" if allpass else "Partially demonstrated",
         "Physics-based prediction and three-detector anomaly monitoring both run inside the closed loop and in the dashboard.",
         "Thresholds not tuned on real faults; faults are synthetic and injected into the same simulator; false-alarm rate is non-zero."),
        ("Bidirectional data flow",
         "Asset->twin telemetry and twin->asset commands exchanged in software and logged",
         "MessageBus, PlantEmulator, DigitalTwin; telemetry_log_S1.csv, command_log_S1.csv; tests 29-31; Figure 1",
         f"{len(ev['runs']['S1 Default (flows 0.1 L/min)'].telemetry_log)} telemetry and {len(ev['runs']['S1 Default (flows 0.1 L/min)'].command_log)} command messages in scenario S1; target change alters asset response (test 30)",
         "Demonstrated" if allpass else "Partially demonstrated",
         "Real exchange of JSON messages between two separate objects, with the asset responding to commands.",
         "Both endpoints are simulated in one Python process; no real sensors, network, latency, dropouts or actuator hardware."),
        ("Dashboard functionality",
         "Working controls, run button, graphs, metrics, alerts, calibration information",
         "build_dashboard(), dashboard_update(); notebook last section; tests 38-40",
         "All 14 controls feed the model; tests confirm every listed control changes the output and the Run handler updates all tabs (with stand-in widgets)",
         "Partially demonstrated",
         "Backend and widget wiring are tested; the real ipywidgets interface must be opened in Colab by the user.",
         "Real ipywidgets rendering was not executed in this build environment (ipywidgets not installed there)."),
        ("Reproducibility",
         "Same inputs give same outputs; files and instructions allow rerun",
         "README.md; requirements.txt; fixed seeds; test 02",
         "Identical results for identical seeds (test 02); synthetic dataset seed 42; notebook generated from the tested source file",
         "Demonstrated" if allpass else "Partially demonstrated",
         "Seeded random numbers, a short dependency list (requirements.txt) and a ZIP with all files.",
         "Colab run-through is for the user to confirm; library versions in Colab may differ slightly."),
        ("Limitations and readiness for real-world use",
         "Honest statement of what is not yet established",
         "Report sections 'Limitations' and 'Discussion'",
         "No real tank, sensors or data; calibration and validation are on synthetic data only",
         "Not demonstrated",
         "The system is a software simulation and learning prototype. It is NOT ready to control real equipment.",
         "Needs real measurements, sensor-accuracy data, hardware interface, safety review and independent validation."),
    ]
    df = pd.DataFrame(rows, columns=["Area", "Criterion", "Evidence (location)", "Measurable result", "Status", "Justification", "Remaining limitations"])
    df["Score (0-2)"] = df.Status.map({"Demonstrated": 2, "Partially demonstrated": 1, "Not demonstrated": 0})
    return df


def requirement_checklist(ev):
    """Requirement-to-evidence checklist for the official project overview."""
    rows = [
        ("Asset with at least two coupled states", "digital_twin_tank.py: step_tank() (T and h); report Section 5", "Tests 04, 07, 08, 09, 10; Figures 3-5", "Complete (software simulation)"),
        ("At least one controllable input", "Heater command u (twin->asset); also flows, inlet T are scenario inputs", "Test 30; command_log_S1.csv", "Complete"),
        ("Both directions of data flow", "MessageBus, PlantEmulator.publish_telemetry / receive_command, DigitalTwin", "Tests 29-31; telemetry/command logs; Figure 1", "Complete in software; no real hardware"),
        ("Tested simulation core", "tests/test_digital_twin_tank.py (40 tests)", "results/tables/test_results.csv", "Complete" if ev.get("tests") is not None and (ev["tests"].result == "PASS").all() else "Partial"),
        ("Calibrated parameters", "calibration_workflow() on synthetic benchmark dataset", "Figure 7; cal_params_table.csv", "Partial: synthetic data only, not a real tank"),
        ("At least two intelligence-layer elements", "predict_time_to_target(); AnomalyDetector (+ recommendations)", "Tests 21-28; Figure 6; scenario table", "Complete in simulation"),
        ("A dashboard", "build_dashboard() in notebook (IPyWidgets)", "Tests 38-40; notebook final section", "Partial: widget rendering to be confirmed in Colab"),
        ("Completed credibility scorecard", "build_scorecard(); credibility_scorecard.csv/.md; report Section 14", "results/tables/scorecard.csv", "Complete (honest statuses: mixed)"),
        ("Individual or pair work", "Author field in report must be filled by the student", "Report title page", "Unavailable to me: student must enter names"),
        ("Real measurements for calibration", "None supplied; load_reference_csv() is ready for real data", "-", "Unavailable (flagged)"),
    ]
    return pd.DataFrame(rows, columns=["Requirement", "Implementation", "Evidence", "Status"])

## 9. Verify the original report is reproduced
The original one-state simulation (seed 0) is rerun and compared with the numbers in the original report.

In [ ]:
import pandas as pd
base = original_one_state_baseline()
rep = REPORT_BASELINE
chk = pd.DataFrame([
    ("Final temperature (°C)", rep["final_T"], round(base["T"].iloc[-1], 1)),
    ("Heater ON (% of steps)", rep["heater_on_pct"], round(100 * base["u"].mean(), 1)),
    ("Heater ON time (min)", rep["heater_on_min"], round(base["u"].sum() * 10 / 60, 1)),
    ("Temperature at 30 min (°C)", rep["T30"], round(base["T"].iloc[179], 1)),
    ("Temperature at 60 min (°C)", rep["T60"], round(base["T"].iloc[359], 1)),
    ("Temperature at 90 min (°C)", rep["T90"], round(base["T"].iloc[539], 1))],
    columns=["Quantity", "Original report", "Re-run now"])
display(chk) if "display" in globals() else print(chk)

## 10. Run the automated tests
The test code below is the same as `tests/test_digital_twin_tank.py`. Each test's acceptance criterion is shown in the table.

In [ ]:
import sys, types
_m = types.ModuleType("digital_twin_tank"); _m.__dict__.update(globals()); sys.modules["digital_twin_tank"] = _m  # lets the tests import the code above

In [ ]:
"""Automated tests for the heated-tank digital twin.
Run with:  python -m unittest discover -s tests -v      (or:  python -m pytest tests)
Each test docstring states its ACCEPTANCE CRITERION. Tests use physical limiting cases with
closed-form answers where possible, so they check behaviour rather than repeating the code."""
import math
import os
import sys
import tempfile
import unittest
from dataclasses import replace

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import digital_twin_tank as dt

P0 = replace(dt.NOMINAL_TWIN, evap_scale=0.0)           # nominal, evaporation switched off
DT = 10.0


def run_steps(p, T0, h0, n, u=1, q_in=0.0, q_out=0.0, T_in=15.0, T_amb=20.0, dt_=DT):
    T, h, diag = T0, h0, []
    for _ in range(n):
        o = dt.step_tank(T, h, u, q_in, q_out, T_in, T_amb, p, dt_)
        T, h = o["T"], o["h"]
        diag.append(o)
    return T, h, diag


class TestSimulationCore(unittest.TestCase):
    def test_01_duration_and_step_count(self):
        """180 min at 10 s -> 1080 steps: 1081 samples, last time 10800 s, 1080 commands, 1081 telemetry messages."""
        r = dt.run_twin_simulation(dt.Scenario())
        self.assertEqual(len(r.df), 1081)
        self.assertAlmostEqual(r.df.time_s.iloc[-1], 10800.0)
        self.assertEqual(len(r.command_log), 1080)
        self.assertEqual(len(r.telemetry_log), 1081)
        self.assertTrue(np.allclose(np.diff(r.df.time_s), DT))

    def test_02_reproducible_with_seed(self):
        """Same seed -> identical results; different seed -> different sensor noise."""
        a = dt.run_twin_simulation(dt.Scenario(seed=3)).df
        b = dt.run_twin_simulation(dt.Scenario(seed=3)).df
        c = dt.run_twin_simulation(dt.Scenario(seed=4)).df
        pd.testing.assert_frame_equal(a, b)
        self.assertFalse(np.allclose(a.T_meas, c.T_meas))

    def test_03_original_baseline_reproduced(self):
        """Original one-state code reproduces the report: final 55.3 degC, 67.6 % ON, 121.7 min ON."""
        d = dt.original_one_state_baseline()
        self.assertAlmostEqual(d["T"].iloc[-1], 55.28, places=1)
        self.assertAlmostEqual(100 * d["u"].mean(), 67.6, places=1)
        self.assertAlmostEqual(d["u"].sum() * 10 / 60, 121.7, places=1)

    def test_04_extension_reduces_to_original_equation(self):
        """No flow, no evaporation, wall mass 0, h fixed by construction: T update equals the original equation to 1e-9."""
        T, h = 20.0, 0.5
        Tref = 20.0
        for _ in range(200):
            o = dt.step_tank(T, h, 1, 0, 0, 15, 20, P0, DT)
            Tref = Tref + DT / 420000.0 * (3000.0 * 1 - 25.0 * (Tref - 20.0))
            T, h = o["T"], o["h"]
        self.assertAlmostEqual(T, Tref, places=9)
        self.assertAlmostEqual(h, 0.5, places=12)
        self.assertAlmostEqual(dt.thermal_capacity(0.5, P0), 420000.0)
        self.assertAlmostEqual(dt.heat_loss_coeff(0.5, P0), 25.0)

    def test_05_heater_off_cools_to_ambient_exponentially(self):
        """Heater OFF, no flow: T(t)=Tamb+(T0-Tamb)exp(-kt/C) within 0.05 degC over 2 h; monotone decrease."""
        T, h, diag = run_steps(P0, 60.0, 0.5, 720, u=0)
        exact = 20.0 + 40.0 * math.exp(-25.0 * 7200 / 420000.0)
        self.assertAlmostEqual(T, exact, delta=0.05)
        Ts = [60.0] + [60.0]
        T1, _, _ = run_steps(P0, 60.0, 0.5, 1, u=0)
        self.assertLess(T1, 60.0)
        self.assertGreater(T, 20.0)

    def test_06_heating_response_matches_closed_form(self):
        """Heater ON, no flow: T(t)=Teq+(T0-Teq)exp(-t/tau), Teq=Tamb+P/k=140 degC (unbounded), checked at 1 h within 0.05 degC; first-step rate = P/C."""
        T, h, _ = run_steps(P0, 20.0, 0.5, 360, u=1)
        Teq, tau = 20.0 + 3000.0 / 25.0, 420000.0 / 25.0
        self.assertAlmostEqual(T, Teq + (20.0 - Teq) * math.exp(-3600 / tau), delta=0.05)
        T1, _, _ = run_steps(P0, 20.0, 0.5, 1, u=1)
        self.assertAlmostEqual((T1 - 20.0) / DT, 3000.0 / 420000.0, places=9)

    def test_07_level_changes_thermal_capacity(self):
        """Half the water -> half the capacity -> twice the temperature rise per step (at T = Tamb, so no loss)."""
        self.assertAlmostEqual(dt.thermal_capacity(0.25, P0) * 2, dt.thermal_capacity(0.5, P0))
        T_half, _, _ = run_steps(P0, 20.0, 0.25, 1)
        T_full, _, _ = run_steps(P0, 20.0, 0.50, 1)
        self.assertAlmostEqual((T_half - 20.0) / (T_full - 20.0), 2.0, places=6)
        # wall mass adds capacity that does not depend on level
        pw = replace(P0, c_wall=50000.0)
        self.assertAlmostEqual(dt.thermal_capacity(0.5, pw), 470000.0)

    def test_08_evaporation_depends_on_temperature(self):
        """Evaporation rate rises with water temperature, is 0 when water is at room temperature in saturated air, and the level falls faster when hot."""
        p = dt.NOMINAL_TWIN
        self.assertLess(dt.evap_rate(30, 20, p), dt.evap_rate(50, 20, p))
        self.assertLess(dt.evap_rate(50, 20, p), dt.evap_rate(70, 20, p))
        self.assertEqual(dt.evap_rate(20, 20, replace(p, rel_humidity=1.0)), 0.0)
        _, h_cold, _ = run_steps(replace(p, k_scale=0), 30.0, 0.5, 360, u=0)
        _, h_hot, _ = run_steps(replace(p, k_scale=0), 70.0, 0.5, 360, u=0)
        self.assertLess(h_hot, h_cold)
        self.assertLess(h_cold, 0.5)

    def test_09_mass_balance_closes(self):
        """m_final - m_initial = sum(inflow - outflow - evaporation - overflow) to 1e-9 kg, including a run that overflows."""
        p = dt.NOMINAL_TWIN
        T, h, diag = run_steps(p, 40.0, 0.78, 400, u=1, q_in=6.0, q_out=1.0)
        m0, m1 = dt.RHO * p.area * 0.78, dt.RHO * p.area * h
        net = sum(d["m_in"] - d["m_out"] - d["m_ev"] - d["m_ov"] for d in diag)
        self.assertAlmostEqual(m1 - m0, net, places=9)
        self.assertGreater(sum(d["m_ov"] for d in diag), 0.0)
        self.assertLessEqual(h, p.h_max + 1e-12)

    def test_10_energy_balance_closes(self):
        """Enthalpy change = heater + inflow - outflow - evaporation - overflow - wall loss to 1e-6 relative (no clipping)."""
        p = dt.NOMINAL_TWIN
        T0, h0 = 35.0, 0.5
        T, h, diag = run_steps(p, T0, h0, 500, u=1, q_in=2.0, q_out=1.5, T_in=15.0)
        U0 = (dt.RHO * p.area * h0 * dt.CP + p.c_wall) * T0
        U1 = (dt.RHO * p.area * h * dt.CP + p.c_wall) * T
        src = sum(d["e_heat"] + d["e_in"] - d["e_out"] - d["e_ev"] - d["e_ov"] - d["e_loss"] for d in diag)
        self.assertFalse(any(d["clipped"] for d in diag))
        self.assertAlmostEqual((U1 - U0) / src, 1.0, places=6)

    def test_11_mixing_limit_matches_analytic(self):
        """Heater OFF, no loss, equal inflow/outflow of 20 degC water: T -> T_in as exp(-qt/m); error < 0.05 degC after 1 h."""
        p = replace(P0, k_scale=0.0)
        q = 6.0   # L/min = 0.1 kg/s
        T, h, _ = run_steps(p, 60.0, 0.5, 360, u=0, q_in=q, q_out=q, T_in=20.0)
        exact = 20.0 + 40.0 * math.exp(-(q / 60.0) * 3600 / 100.0)
        self.assertAlmostEqual(T, exact, delta=0.5)   # first-order Euler on a 1000 s time constant
        self.assertAlmostEqual(h, 0.5, places=9)

    def test_12_minimum_level_handling(self):
        """Large outflow from low level: level never below h_min, outlet throttles, heater inhibited below dry-fire level, no NaN."""
        p = dt.NOMINAL_TWIN
        T, h = 50.0, 0.20
        inhibited, saw_low = True, False
        for _ in range(600):
            h_before = h
            o = dt.step_tank(T, h, 1, 0.0, 50.0, 15.0, 20.0, p, DT)
            if h_before < p.h_heater_min:      # interlock depends on the level BEFORE the step
                saw_low = True
                inhibited = inhibited and o["u_eff"] == 0.0
            T, h = o["T"], o["h"]
            self.assertGreaterEqual(h, p.h_min - 1e-9)
        self.assertTrue(saw_low and inhibited)
        self.assertAlmostEqual(h, p.h_min, places=6)
        self.assertTrue(np.isfinite(T))

    def test_13_maximum_level_handling(self):
        """Large inflow: level capped at h_max, surplus leaves as overflow, state finite."""
        p = dt.NOMINAL_TWIN
        T, h, diag = run_steps(p, 50.0, 0.79, 300, u=0, q_in=50.0, q_out=0.0, T_in=15.0)
        self.assertAlmostEqual(h, p.h_max, places=9)
        self.assertGreater(sum(d["m_ov"] for d in diag), 100.0)
        self.assertTrue(np.isfinite(T) and 15.0 <= T <= 50.0)

    def test_14_over_temperature_interlock(self):
        """Above 90 degC the heater is forced OFF regardless of command."""
        o = dt.step_tank(91.0, 0.5, 1, 0, 0, 15, 20, dt.NOMINAL_TWIN, DT)
        self.assertEqual(o["u_eff"], 0.0)
        self.assertLess(o["T"], 91.0)

    def test_15_no_nan_or_inf_in_extreme_valid_scenarios(self):
        """Corner-case valid inputs give finite results within 0-100 degC and h_min..h_max."""
        cases = [dict(q_in=10, q_out=0, T0=90, heater_power=5000, duration_min=120),
                 dict(q_in=0, q_out=10, h0=0.11, duration_min=120),
                 dict(T0=1, T_in=60, q_in=10, q_out=10, T_amb=0, heater_power=500),
                 dict(noise_sd=1.5, controller="onoff_raw", target=85, duration_min=300)]
        for c in cases:
            r = dt.run_twin_simulation(dt.Scenario(**c))
            self.assertTrue(r.metrics["all_finite"], c)
            self.assertTrue(r.df.T_true.between(0, 100).all(), c)
            self.assertTrue(r.df.h_true.between(dt.NOMINAL_TWIN.h_min - 1e-9, dt.NOMINAL_TWIN.h_max + 1e-9).all(), c)

    def test_16_invalid_inputs_rejected(self):
        """Impossible inputs raise ValueError instead of running."""
        for bad in (dict(h0=0.0), dict(T0=-5), dict(q_in=-1), dict(heater_power=-10), dict(duration_min=0),
                    dict(target=200), dict(dt=100.0)):
            with self.assertRaises(ValueError, msg=str(bad)):
                dt.validate_scenario(dt.Scenario(**bad))


class TestControlAndMetrics(unittest.TestCase):
    def test_17_onoff_rule_on_raw_reading(self):
        """'onoff_raw': command is 1 exactly when the noisy reading is below the target (all 1080 steps)."""
        r = dt.run_twin_simulation(dt.Scenario(controller="onoff_raw"))
        tel, cmd = r.telemetry_log.iloc[:-1], r.command_log
        self.assertTrue(np.array_equal(cmd.heater_cmd.to_numpy(), (tel.T_meas.to_numpy() < 55.0).astype(int)))

    def test_18_hysteresis_rule(self):
        """Twin controller: ON when estimate <= target-0.5, OFF when >= target+0.5, otherwise unchanged; fewer switches than raw rule."""
        s = dt.Scenario(hysteresis=1.0)
        r = dt.run_twin_simulation(s)
        d = r.df.iloc[:-1]
        self.assertTrue((d.heater_cmd[d.T_hat <= 54.5] == 1).all())
        self.assertTrue((d.heater_cmd[d.T_hat >= 55.5] == 0).all())
        raw = dt.run_twin_simulation(dt.Scenario(controller="onoff_raw"))
        self.assertLess(r.metrics["n_switches"], raw.metrics["n_switches"])

    def test_19_mae_rmse_heater_time(self):
        """MAE/RMSE/ON-fraction on hand-computed values."""
        a, b = [1, 2, 3, 4], [1, 3, 5, 4]
        self.assertAlmostEqual(dt.mae(a, b), 0.75)
        self.assertAlmostEqual(dt.rmse(a, b), math.sqrt(5 / 4))
        self.assertAlmostEqual(dt.heater_on_fraction([1, 1, 0, 0, 0, 1, 0, 0]), 0.375)
        r = dt.run_twin_simulation(dt.Scenario(duration_min=60))
        on = r.df.heater_state.iloc[1:].mean() * 100
        self.assertAlmostEqual(r.metrics["heater_on_pct"], on, places=9)
        self.assertAlmostEqual(r.metrics["heater_on_min"], on / 100 * 60, places=6)

    def test_20_higher_target_needs_more_time_and_more_power_less(self):
        """Monotonic behaviour: a higher target takes longer; more heater power reaches 55 degC sooner."""
        t = lambda **k: dt.run_twin_simulation(dt.Scenario(duration_min=300, **k)).metrics["time_to_target_actual_min"]
        self.assertLess(t(target=45), t(target=55))
        self.assertLess(t(heater_power=4000), t(heater_power=3000))


class TestIntelligenceLayer(unittest.TestCase):
    def test_21_time_to_target_matches_closed_form(self):
        """No flow/evaporation: predicted time equals t=-tau ln((target-Teq)/(T0-Teq)) within 1 %."""
        pred = dt.predict_time_to_target(20.0, 0.5, 55.0, 0, 0, 15, 20, P0)
        Teq, tau = 20.0 + 3000 / 25.0, 420000 / 25.0
        exact = -tau * math.log((55 - Teq) / (20 - Teq))
        self.assertEqual(pred["status"], "ok")
        self.assertAlmostEqual(pred["seconds"] / exact, 1.0, delta=0.01)

    def test_22_time_to_target_unavailable_cases(self):
        """Unreachable target -> status 'unavailable' with a reason; already above target -> 0 s."""
        weak = dt.predict_time_to_target(20.0, 0.5, 55.0, 0, 0, 15, 20, replace(P0, heater_power=300.0))
        self.assertEqual(weak["status"], "unavailable")
        self.assertIn("not reachable", weak["reason"])
        low = dt.predict_time_to_target(20.0, 0.12, 55.0, 0, 0, 15, 20, P0)
        self.assertEqual(low["status"], "unavailable")
        self.assertIn("dry-fire", low["reason"])
        self.assertEqual(dt.predict_time_to_target(60.0, 0.5, 55.0, 0, 0, 15, 20, P0)["seconds"], 0.0)

    def test_23_prediction_recalculated_when_conditions_change(self):
        """More outflow of cold make-up water or lower power lengthens the predicted time."""
        base = dt.run_twin_simulation(dt.Scenario()).metrics["predicted_ttt_initial_min"]
        more = dt.run_twin_simulation(dt.Scenario(q_in=0.5, q_out=0.5, duration_min=300)).metrics["predicted_ttt_initial_min"]
        self.assertGreater(more, base)

    def test_24_prediction_accuracy_in_default_run(self):
        """Prediction made at t=0 is within 5 min of the realised time to target in the default simulated asset."""
        m = dt.run_twin_simulation(dt.Scenario()).metrics
        self.assertLess(abs(m["ttt_error_min"]), 5.0)

    def test_25_injected_anomalies_detected(self):
        """Injected TEST anomalies are detected: spike within 2 samples; bias within 5 min; heater degradation within 15 min."""
        r = dt.run_twin_simulation(dt.Scenario(anomaly_type="spike", anomaly_start_min=90))
        first = min(a["t_start"] for a in r.alerts if a["type"] == "spike")
        self.assertLessEqual(first - 90 * 60, 2 * DT)
        r = dt.run_twin_simulation(dt.Scenario(anomaly_type="bias", anomaly_start_min=90))
        first = min(a["t_start"] for a in r.alerts if a["type"] in ("spike", "drift"))
        self.assertLessEqual(first - 90 * 60, 300)
        r = dt.run_twin_simulation(dt.Scenario(anomaly_type="heater_degradation", anomaly_start_min=30))
        first = min(a["t_start"] for a in r.alerts if a["type"] == "rate")
        self.assertLessEqual(first - 30 * 60, 900)

    def test_26_false_alarm_rate_without_anomaly(self):
        """30 normal runs (seeds 0-29, 180 min each) raise at most 6 alerts in total (documented false-alarm rate <= 0.2 per run)."""
        n = sum(len(dt.run_twin_simulation(dt.Scenario(seed=s)).alerts) for s in range(30))
        self.assertLessEqual(n, 6)

    def test_27_spike_rejected_by_twin(self):
        """A spike reading is rejected, so the twin estimate is much closer to the truth than the raw reading during the spike."""
        r = dt.run_twin_simulation(dt.Scenario(anomaly_type="spike", anomaly_start_min=90))
        d = r.df[(r.df.time_s >= 90 * 60) & (r.df.time_s < 90 * 60 + 30)]
        self.assertGreaterEqual(r.metrics["n_rejected"], 1)
        self.assertLess((d.T_hat - d.T_true).abs().max(), 1.0)
        self.assertGreater((d.T_meas - d.T_true).abs().min(), 3.0)

    def test_28_recommendations_generated(self):
        """Impossible-target scenario yields a 'not reachable' recommendation."""
        r = dt.run_twin_simulation(dt.Scenario(heater_power=500, q_in=1.0, q_out=1.0, duration_min=60, target=70))
        self.assertTrue(any("not reachable" in x.lower() for x in r.recommendations))


class TestDataFlow(unittest.TestCase):
    def test_29_both_directions_are_real_and_logged(self):
        """Telemetry (A) and commands (B) pass through the bus as JSON; the asset's heater state follows the previous command; the bus queues are empty at the end."""
        r = dt.run_twin_simulation(dt.Scenario(duration_min=30))
        self.assertTrue({"T_meas", "h_meas", "heater_state", "q_in", "q_out", "T_in"} <= set(r.telemetry_log.columns))
        self.assertTrue({"heater_cmd", "target", "mode", "reason"} <= set(r.command_log.columns))
        self.assertEqual(len(r.bus.to_twin), 0)
        self.assertEqual(len(r.bus.to_asset), 0)
        self.assertTrue(np.array_equal(r.df.heater_state.iloc[1:].to_numpy(), r.df.heater_cmd.iloc[:-1].to_numpy()))

    def test_30_command_changes_asset_behaviour(self):
        """Direction B has effect: changing the twin's target changes the commands and the asset's true temperature."""
        a = dt.run_twin_simulation(dt.Scenario(target=40.0, duration_min=150))
        b = dt.run_twin_simulation(dt.Scenario(target=60.0, duration_min=150))
        self.assertGreater(b.df.T_true.iloc[-1] - a.df.T_true.iloc[-1], 10.0)
        self.assertNotEqual(list(a.command_log.target.unique()), list(b.command_log.target.unique()))

    def test_31_twin_uses_only_telemetry(self):
        """Direction A: a hand-made telemetry message initialises the twin state; truth is not accessible to the twin."""
        s = dt.Scenario()
        bus = dt.MessageBus()
        twin = dt.DigitalTwin(s, dt.NOMINAL_TWIN, bus)
        bus.publish_telemetry(dict(t=0.0, T_meas=33.3, h_meas=0.42, heater_state=0.0, q_in=0.1, q_out=0.1, T_in=15.0))
        twin.process_telemetry()
        self.assertEqual((twin.T_hat, twin.h_hat), (33.3, 0.42))
        self.assertFalse(hasattr(twin, "plant"))


class TestCalibration(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.cal = dt.calibration_workflow(apply=False)

    def test_32_dataset_is_labelled_synthetic_and_split(self):
        """Dataset carries a SYNTHETIC label, a 60/40 calibration/validation split, a uniform 10 s grid and 241 min..."""
        df = self.cal["df"]
        self.assertTrue(df.data_origin.str.contains("SYNTHETIC").all())
        frac = (df.split == "calibration").mean()
        self.assertAlmostEqual(frac, 0.6, delta=0.01)
        self.assertTrue(np.allclose(np.diff(df.time_s), DT))
        self.assertTrue(np.isfinite(df[dt.DATA_COLUMNS].to_numpy()).all())

    def test_33_calibration_improves_validation_fit(self):
        """On held-out data the calibrated twin has temperature RMSE < 0.5 degC and at least 50 % lower than the nominal twin."""
        v_cal, v_nom = self.cal["calibrated"]["validation"]["rmse_T"], self.cal["nominal"]["validation"]["rmse_T"]
        self.assertLess(v_cal, 0.5)
        self.assertLess(v_cal, 0.5 * v_nom)

    def test_34_estimates_within_bounds_and_plausible(self):
        """Estimates are strictly inside their bounds and heater efficiency is within 10 % of the benchmark value."""
        for k, v in self.cal["theta"].items():
            lo, hi = dt.BOUNDS[k]
            self.assertTrue(lo < v < hi, k)
        ref = dt.reference_values()
        self.assertLess(abs(self.cal["theta"]["heater_eff"] - ref["heater_eff"]) / ref["heater_eff"], 0.10)

    def test_35_no_validation_leakage(self):
        """Corrupting the VALIDATION rows (+10 degC) does not change the fitted parameters; corrupting calibration rows does."""
        df = self.cal["df"].copy()
        df.loc[df.split == "validation", "T_meas_C"] += 10.0
        c2 = dt.calibration_workflow(df, apply=False)
        for k in self.cal["theta"]:
            self.assertAlmostEqual(self.cal["theta"][k], c2["theta"][k], places=6)
        df2 = self.cal["df"].copy()
        df2.loc[df2.split == "calibration", "T_meas_C"] += 2.0
        c3 = dt.calibration_workflow(df2, apply=False)
        self.assertGreater(abs(self.cal["theta"]["heater_eff"] - c3["theta"]["heater_eff"]), 1e-3)

    def test_36_csv_loader_for_real_data(self):
        """Real-data loader: fills short gaps and reports them, rejects missing columns, bad units and long gaps."""
        df = self.cal["df"][dt.DATA_COLUMNS].copy()
        with tempfile.TemporaryDirectory() as d:
            p = os.path.join(d, "x.csv")
            df2 = df.copy(); df2.loc[[10, 11], "T_meas_C"] = np.nan; df2.to_csv(p, index=False)
            out, rep = dt.load_reference_csv(p)
            self.assertEqual(rep["missing_values_filled"], 2)
            self.assertEqual(out.data_origin.iloc[0], "REAL_USER_DATA")
            df.drop(columns=["h_meas_m"]).to_csv(p, index=False)
            with self.assertRaises(ValueError):
                dt.load_reference_csv(p)
            df3 = df.copy(); df3["T_meas_C"] = df3["T_meas_C"] + 273.15; df3.to_csv(p, index=False)
            with self.assertRaises(ValueError):
                dt.load_reference_csv(p)
            df4 = df.copy(); df4.loc[20:30, "T_meas_C"] = np.nan; df4.to_csv(p, index=False)
            with self.assertRaises(ValueError):
                dt.load_reference_csv(p)

    def test_37_uncertainty_and_sensitivity_run(self):
        """Monte Carlo gives a non-zero spread; sensitivity: +10 % heater power shortens time to target, +10 % loss lengthens it."""
        unc = dt.uncertainty_analysis(self.cal, n=20)
        self.assertGreater(unc["noise_only"]["t_target_min"][1], 0.0)
        self.assertGreater(unc["band_T"].std(axis=0).max(), 0.0)
        tab = dt.sensitivity_analysis(self.cal)["table"]
        g = lambda par, ch: tab[(tab.parameter == par) & (tab.change == ch)].t_target_min.iloc[0]
        self.assertLess(g("heater_power", "+10%"), g("heater_power", "-10%"))
        self.assertGreater(g("k_scale", "+10%"), g("k_scale", "-10%"))


class StubWidget:
    """Minimal stand-in for ipywidgets (real widgets are not installed in the test environment)."""
    def __init__(self, *a, **k):
        self.__dict__.update(k)
        self.children, self.value = k.get("children", []), k.get("value")
        self._handlers, self.titles, self.log = [], {}, []
    def on_click(self, f): self._handlers.append(f)
    def _click(self):
        for f in self._handlers: f(self)
    def set_title(self, i, t): self.titles[i] = t
    def __enter__(self): return self
    def __exit__(self, *a): return False


class StubWidgets:
    FloatSlider = Dropdown = IntText = Button = HTML = Output = VBox = Tab = StubWidget
    class Layout(dict):
        def __init__(self, **k): super().__init__(k)


class TestDashboard(unittest.TestCase):
    def values(self, **over):
        v = {k: d for k, _, _, _, _, d in dt.CONTROL_SPECS}
        v.update(controller="twin_hysteresis", anomaly_type="none", anomaly_start_min=90, seed=0)
        v.update(over)
        return v

    def test_38_dashboard_backend_responds_to_each_input(self):
        """Changing target, power, initial temperature, flows, noise, duration, level or ambient changes the outputs."""
        base = dt.dashboard_update(self.values())
        self.assertTrue(base["ok"])
        for k, new in dict(target=45, heater_power=2000, T0=40, q_out=1.0, noise_sd=0.8, duration_min=90, h0=0.3, T_amb=10, T_in=40, q_in=1.0).items():
            r = dt.dashboard_update(self.values(**{k: new}))
            self.assertTrue(r["ok"], k)
            changed = (r["status"] != base["status"]) or (r["metrics"].Value.tolist() != base["metrics"].Value.tolist())
            self.assertTrue(changed, f"{k} had no effect")
        r = dt.dashboard_update(self.values(target=40))
        self.assertIn("40", r["status"])
        self.assertAlmostEqual(r["result"].df.T_true.iloc[-1], 40.0, delta=1.5)

    def test_39_dashboard_rejects_bad_input(self):
        """Out-of-range input returns ok=False with a message and does not crash."""
        r = dt.dashboard_update(self.values(heater_power=-5))
        self.assertFalse(r["ok"])
        self.assertIn("Invalid input", r["status"])

    def test_40_widget_wiring_with_stub_widgets(self):
        """(Stub-based) Clicking Run reads the control values, reruns the model and updates the status text and all tabs."""
        shown = []
        box = dt.build_dashboard(widgets=StubWidgets, display_fn=lambda *a, **k: shown.append(a), clear_fn=lambda **k: None)
        self.assertEqual(len(box.controls), len(dt.CONTROL_SPECS) + 4)
        box.controls["target"].value = 42.0
        box.run_button._click()
        self.assertIn("42", box.status_widget.value)
        n1 = len(shown)
        self.assertGreater(n1, 4)
        box.controls["target"].value = 50.0
        box.controls["anomaly_type"].value = "spike"
        box.run_button._click()
        self.assertIn("50", box.status_widget.value)
        self.assertIn("TEST SCENARIO", box.status_widget.value)
        self.assertGreater(len(shown), n1)
        box.controls["heater_power"].value = -1
        box.run_button._click()
        self.assertIn("Invalid", box.status_widget.value)


def run_all(verbose=False):
    """Run the suite and return a DataFrame: test, result (PASS/FAIL/ERROR), acceptance criterion."""
    import io
    suite = unittest.TestSuite()
    for obj in list(globals().values()):          # works both as a file and inside a notebook
        if isinstance(obj, type) and issubclass(obj, unittest.TestCase) and obj is not unittest.TestCase:
            suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(obj))
    rows = []

    class R(unittest.TextTestResult):
        def addSuccess(s, t): super().addSuccess(t); rows.append((t, "PASS", None))
        def addFailure(s, t, e): super().addFailure(t, e); rows.append((t, "FAIL", e))
        def addError(s, t, e): super().addError(t, e); rows.append((t, "ERROR", e))

    import contextlib
    with contextlib.redirect_stdout(io.StringIO()):          # dashboard test prints; keep the summary clean
        unittest.TextTestRunner(stream=io.StringIO() if not verbose else sys.stderr, resultclass=R, verbosity=0).run(suite)
    out = [dict(test=t._testMethodName, result=r, criterion=(t._testMethodDoc or "").strip().split("\n")[0]) for t, r, _ in rows]
    return pd.DataFrame(out).sort_values("test").reset_index(drop=True)


In [ ]:
tests_df = run_all()
print(tests_df.result.value_counts().to_dict())
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 250)
display(tests_df)

## 11. Synthetic reference data, calibration, validation, uncertainty and all experiments
This cell reruns everything and saves figures/tables to `results/`. It takes about a minute.

In [ ]:
import os
os.makedirs("results", exist_ok=True); os.makedirs("data", exist_ok=True)
ev = collect_evidence(outdir="results", tests_df=tests_df)
print("done")

In [ ]:
from IPython.display import display, Image
def show(name): display(Image(filename=f"results/figures/{name}"))
print("Calibrated parameters vs SYNTHETIC benchmark values"); display(ev["cal_params_table"])
print("Calibration / validation errors (synthetic data)"); display(ev["cal_metrics_table"])
show("fig7_calibration.png")

In [ ]:
show("fig8_uncertainty.png"); show("fig9_sensitivity.png")
print("Monte Carlo summaries (mean, std, 5th, 95th percentile):")
print("noise only :", ev["unc"]["noise_only"]["t_target_min"]); print("noise+param:", ev["unc"]["both"]["t_target_min"])

In [ ]:
print("Scenario results"); display(ev["scenario_table"])
show("fig2_baseline_vs_extended.png"); show("fig3_default_run.png"); show("fig4_drain_run.png"); show("fig5_overflow_run.png"); show("fig10_controller_comparison.png")

In [ ]:
print("Injected TEST anomalies (synthetic)"); display(ev["anomaly_table"]); print(ev["false_alarms"], ev["ttt_error"])
show("fig6_anomaly_tests.png"); show("fig1_architecture.png")

In [ ]:
print("Telemetry (Direction A) - first rows"); display(ev["runs"]["S1 Default (flows 0.1 L/min)"].telemetry_log.head())
print("Commands (Direction B) - first rows"); display(ev["runs"]["S1 Default (flows 0.1 L/min)"].command_log.head())

## 12. Completed credibility scorecard and requirement checklist

In [ ]:
pd.set_option("display.max_colwidth", 300)
display(ev["scorecard"]); print("Total score:", int(ev["scorecard"]["Score (0-2)"].sum()), "/", 2 * len(ev["scorecard"]))
display(requirement_checklist(ev))

## 13. Interactive dashboard
Change any slider and press **Run / Simulate**. Anything marked *TEST* injects a synthetic anomaly.

In [ ]:
dashboard = build_dashboard(credibility_df=ev["scorecard"])
display(dashboard)
dashboard.on_run()   # run once with default values so the panels are not empty

## 14. Download your results

In [ ]:
import shutil
shutil.make_archive("tank_digital_twin_results", "zip", ".", "results")
try:
    from google.colab import files
    files.download("tank_digital_twin_results.zip")
except ImportError:
    print("Not running in Colab: find tank_digital_twin_results.zip in the working folder.")